In [3]:

from pathlib import Path
import zipfile

zip_path = Path("/content/week2_aligned_omics_package.zip")
output_dir = Path("/content/week3_data")

# Confirm the ZIP exists
if not zip_path.exists():
    print("ZIP not found. Files in /content:")
    for file in Path("/content").iterdir():
        print("-", file.name)
else:
    output_dir.mkdir(exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(output_dir)

    print("ZIP extracted successfully!")

    print("\nExtracted files:")
    for file in sorted(output_dir.rglob("*")):
        if file.is_file():
            print("-", file.relative_to(output_dir))


ZIP extracted successfully!

Extracted files:
- README.txt
- XY_gene_module_candidates.csv
- cnv_aligned.csv
- expression_aligned.csv
- mutation_aligned.csv
- proteomics_aligned.csv
- sample_metadata.csv
- sex_labels_matched.csv


In [4]:

import pandas as pd
from pathlib import Path

data_dir = Path("/content/week3_data")

# Load the four omics matrices
expression = pd.read_csv(
    data_dir / "expression_aligned.csv",
    index_col=0
)

mutation = pd.read_csv(
    data_dir / "mutation_aligned.csv",
    index_col=0
)

cnv = pd.read_csv(
    data_dir / "cnv_aligned.csv",
    index_col=0
)

proteomics = pd.read_csv(
    data_dir / "proteomics_aligned.csv",
    index_col=0
)

# Load metadata
sample_metadata = pd.read_csv(
    data_dir / "sample_metadata.csv"
)

sex_metadata = pd.read_csv(
    data_dir / "sex_labels_matched.csv"
)

# Verify dataset shapes
print("=== OMICS DATASETS ===")
print("Expression:", expression.shape)
print("Mutation:", mutation.shape)
print("CNV:", cnv.shape)
print("Proteomics:", proteomics.shape)

# Verify all samples align
print("\n=== SAMPLE ALIGNMENT ===")

reference_ids = expression.index.astype(str)

for name, df in {
    "Mutation": mutation,
    "CNV": cnv,
    "Proteomics": proteomics
}.items():
    print(
        name,
        "aligned:",
        df.index.astype(str).equals(reference_ids)
    )

print("\n=== SEX LABELS ===")
print("Sex metadata shape:", sex_metadata.shape)

if "usable_sex" in sex_metadata.columns:
    print(
        "Usable sex labels:",
        sex_metadata["usable_sex"].sum()
    )

print("\nSex metadata columns:")
print(sex_metadata.columns.tolist())


=== OMICS DATASETS ===
Expression: (526, 18025)
Mutation: (526, 1667)
CNV: (526, 23264)
Proteomics: (526, 8498)

=== SAMPLE ALIGNMENT ===
Mutation aligned: True
CNV aligned: True
Proteomics aligned: True

=== SEX LABELS ===
Sex metadata shape: (526, 30)
Usable sex labels: 492

Sex metadata columns:
['Cell_line', 'SIDM', 'Project_Identifier', 'Tissue_type', 'Cancer_type', 'Cancer_subtype', 'normalized_name', 'match_key', 'ModelID', 'CellLineName', 'cvcl_final', 'OncotreeLineage', 'reported_sex', 'inferred_sex', 'status', 'review_decision', 'final_sex', 'label_source', 'label_confidence', 'mislabeled', 'recovered', 'uncertain', 'problematic_line', 'exclude_from_sex_analysis', 'xist_score', 'y_score', 'y_cn', 'amelogenin', 'usable_sex', 'high_confidence_usable']


In [5]:

from pathlib import Path

print("=== SEARCHING FOR DRUG-RESPONSE FILES ===")

search_locations = [
    Path("/content"),
    Path("/content/drive/MyDrive")
]

keywords = [
    "gdsc",
    "drug_response",
    "drugresponse",
    "ic50",
    "dose_response",
    "screening",
    "split",
    "fold"
]

found_files = []

for location in search_locations:
    if not location.exists():
        continue

    for path in location.rglob("*"):
        if not path.is_file():
            continue

        if path.suffix.lower() not in [
            ".csv", ".tsv", ".txt",
            ".xlsx", ".parquet", ".gz"
        ]:
            continue

        if any(
            word in path.name.lower()
            for word in keywords
        ):
            found_files.append(path)

if found_files:
    for path in found_files[:50]:
        print(path)

    print("\nTotal matching files:", len(found_files))
else:
    print("No matching response or split files found.")


=== SEARCHING FOR DRUG-RESPONSE FILES ===
/content/Train_Val_Freeze Splits - Celusaurous .xlsx
/content/GDSC_IC50.csv

Total matching files: 2


In [6]:

import pandas as pd
from pathlib import Path

path = Path("/content/GDSC_IC50.csv")

# Load drug-response matrix
drug_response = pd.read_csv(
    path,
    index_col=0,
    na_values=["NA", ""]
)

print("=== DRUG-RESPONSE MATRIX ===")
print("Shape:", drug_response.shape)

print("\nFirst five drug IDs:")
print(drug_response.index[:5].tolist())

print("\nFirst five cell-line IDs:")
print(drug_response.columns[:5].tolist())

# Count available measurements
print("\nAvailable drug-response measurements:")
print(int(drug_response.notna().sum().sum()))

# Check how many unique cells/drugs are represented
print("Total drugs:", drug_response.index.nunique())
print("Total cell lines:", drug_response.columns.nunique())

# Preview a small portion
display(drug_response.iloc[:5, :5])

# Check whether your existing sex metadata contains
# ACH identifiers that can match these columns
print("\n=== ACH IDENTIFIER OVERLAP ===")

ach_ids = set(
    sex_metadata["ModelID"].dropna().astype(str)
)

response_ids = set(
    drug_response.columns.astype(str)
)

print(
    "Cell lines matching sex metadata:",
    len(ach_ids & response_ids)
)

print(
    "Example matching IDs:",
    sorted(ach_ids & response_ids)[:10]
)


=== DRUG-RESPONSE MATRIX ===
Shape: (266, 969)

First five drug IDs:
['GDSC:1', 'GDSC:1001', 'GDSC:1004', 'GDSC:1005', 'GDSC:1006']

First five cell-line IDs:
['ACH-002137', 'ACH-000474', 'ACH-002089', 'ACH-000956', 'ACH-000948']

Available drug-response measurements:
208734
Total drugs: 266
Total cell lines: 969


,ACH-002137,ACH-000474,ACH-002089,ACH-000956,ACH-000948
GDSC:1,NaN,NaN,NaN,NaN,NaN
GDSC:1001,7.258918,9.131374,10.182594,8.332992,8.597157
GDSC:1004,-3.802467,-5.702659,-4.499864,-4.366928,-4.563040
GDSC:1005,4.146364,4.551663,1.772586,3.263056,4.091980
GDSC:1006,3.171367,-2.014854,-1.818771,0.252367,-1.688469



=== ACH IDENTIFIER OVERLAP ===
Cell lines matching sex metadata: 508
Example matching IDs: ['ACH-000004', 'ACH-000006', 'ACH-000007', 'ACH-000009', 'ACH-000012', 'ACH-000015', 'ACH-000019', 'ACH-000021', 'ACH-000024', 'ACH-000030']


In [7]:

import pandas as pd
import numpy as np
from pathlib import Path

# Build the mapping between your SIDM and ACH identifiers
cell_map = sex_metadata[
    ["SIDM", "ModelID", "final_sex",
     "usable_sex", "label_confidence", "Tissue_type"]
].copy()

cell_map["SIDM"] = cell_map["SIDM"].astype("string")
cell_map["ModelID"] = cell_map["ModelID"].astype("string")

# Retain rows with both identifiers
cell_map = cell_map.dropna(subset=["SIDM", "ModelID"])

# Identify ambiguous mappings before joining
ambiguous_sidm = (
    cell_map.groupby("SIDM")["ModelID"].nunique()
)
ambiguous_ach = (
    cell_map.groupby("ModelID")["SIDM"].nunique()
)

bad_sidm = set(ambiguous_sidm[ambiguous_sidm > 1].index)
bad_ach = set(ambiguous_ach[ambiguous_ach > 1].index)

print("Ambiguous SIDM mappings:", len(bad_sidm))
print("Ambiguous ACH mappings:", len(bad_ach))

cell_map = cell_map[
    ~cell_map["SIDM"].isin(bad_sidm)
    & ~cell_map["ModelID"].isin(bad_ach)
].drop_duplicates(subset=["SIDM", "ModelID"])

# Match response columns and molecular sample IDs
cell_map = cell_map[
    cell_map["SIDM"].isin(expression.index.astype(str))
    & cell_map["ModelID"].isin(drug_response.columns)
].copy()

# Exclude remaining non-unique identifier relationships
cell_map = cell_map[
    ~cell_map["SIDM"].duplicated(keep=False)
    & ~cell_map["ModelID"].duplicated(keep=False)
].copy()

# Keep provisionally usable sex labels
eligible = cell_map[
    cell_map["usable_sex"].astype(str).str.lower() == "true"
].copy()

eligible = eligible[
    eligible["final_sex"].astype(str).str.lower().isin(
        ["male", "female"]
    )
]

# Count observed responses for each eligible cell
response_counts = drug_response.notna().sum(axis=0)

eligible["n_drug_responses"] = (
    eligible["ModelID"]
    .map(response_counts)
    .fillna(0)
    .astype(int)
)

eligible = eligible[
    eligible["n_drug_responses"] > 0
].copy()

print("\n=== MATCHING RESULTS ===")
print("Matched cell lines (before sex filtering):", len(cell_map))
print("Eligible cell lines:", len(eligible))

print("\nSex distribution:")
print(eligible["final_sex"].value_counts())

print("\nTotal observed cell-drug pairs:",
      eligible["n_drug_responses"].sum())

print("\nTissue groups:",
      eligible["Tissue_type"].nunique())

# Save mapping for the model
output_dir = Path("/content/week3_outputs")
output_dir.mkdir(exist_ok=True)

eligible.to_csv(
    output_dir / "eligible_cell_line_mapping.csv",
    index=False
)

print("\nSaved: eligible_cell_line_mapping.csv")


Ambiguous SIDM mappings: 0
Ambiguous ACH mappings: 0

=== MATCHING RESULTS ===
Matched cell lines (before sex filtering): 508
Eligible cell lines: 491

Sex distribution:
final_sex
Male      267
Female    224
Name: count, dtype: int64

Total observed cell-drug pairs: 106153

Tissue groups: 21

Saved: eligible_cell_line_mapping.csv


In [8]:

import numpy as np
import pandas as pd

# Restrict to the eligible cell lines
matched_responses = drug_response[
    eligible["ModelID"].tolist()
]

# Convert to numeric values
matched_responses = matched_responses.apply(
    pd.to_numeric,
    errors="coerce"
)

values = matched_responses.to_numpy()
valid_values = values[np.isfinite(values)]

print("=== MATCHED DRUG-RESPONSE DISTRIBUTION ===")
print("Observed measurements:", len(valid_values))
print("Minimum:", round(float(valid_values.min()), 4))
print("Maximum:", round(float(valid_values.max()), 4))
print("Mean:", round(float(valid_values.mean()), 4))
print("Median:", round(float(np.median(valid_values)), 4))
print("Standard deviation:", round(float(valid_values.std()), 4))

print("\nPercentiles:")
for p in [1, 5, 25, 50, 75, 95, 99]:
    print(
        f"{p}th percentile:",
        round(float(np.percentile(valid_values, p)), 4)
    )

print("\nMissing entries:",
      int(np.isnan(values).sum()))

print("Infinite entries:",
      int(np.isinf(values).sum()))

print("\nDrug coverage:")
coverage = matched_responses.notna().sum(axis=1)

print("Drugs with at least 20 measurements:",
      int((coverage >= 20).sum()))

print("Drugs with at least 100 measurements:",
      int((coverage >= 100).sum()))

print("Drugs with zero measurements:",
      int((coverage == 0).sum()))


=== MATCHED DRUG-RESPONSE DISTRIBUTION ===
Observed measurements: 106153
Minimum: -9.8468
Maximum: 12.9002
Mean: 2.1476
Median: 2.5916
Standard deviation: 2.6834

Percentiles:
1th percentile: -5.6118
5th percentile: -3.2176
25th percentile: 0.7137
50th percentile: 2.5916
75th percentile: 4.0721
95th percentile: 5.6161
99th percentile: 6.7907

Missing entries: 24453
Infinite entries: 0

Drug coverage:
Drugs with at least 20 measurements: 266
Drugs with at least 100 measurements: 266
Drugs with zero measurements: 0


In [9]:

from pathlib import Path
import pandas as pd
import numpy as np

# Build a long-format table from observed drug responses
response_long = (
    matched_responses
    .rename_axis(index="Drug_ID", columns="ModelID")
    .stack()
    .rename("response")
    .reset_index()
)

# Attach molecular sample IDs and sex metadata
lookup = eligible[
    [
        "ModelID",
        "SIDM",
        "final_sex",
        "label_confidence",
        "Tissue_type"
    ]
].copy()

response_pairs = response_long.merge(
    lookup,
    on="ModelID",
    how="inner",
    validate="many_to_one"
)

# Keep relevant columns
response_pairs = response_pairs[
    [
        "SIDM",
        "ModelID",
        "Drug_ID",
        "response",
        "final_sex",
        "label_confidence",
        "Tissue_type"
    ]
].copy()

# Validate
print("=== FINAL OBSERVATION TABLE ===")
print("Total observations:", len(response_pairs))
print("Unique cell lines:", response_pairs["SIDM"].nunique())
print("Unique drugs:", response_pairs["Drug_ID"].nunique())

print("\nResponses by sex:")
print(
    response_pairs.groupby("final_sex")
    .agg(
        observations=("response", "size"),
        cell_lines=("SIDM", "nunique")
    )
)

print("\nDuplicate cell-drug pairs:",
      response_pairs.duplicated(["SIDM", "Drug_ID"]).sum())

print("\nMissing responses:",
      response_pairs["response"].isna().sum())

# Export
output_dir = Path("/content/week3_outputs")
output_dir.mkdir(exist_ok=True)

response_pairs.to_csv(
    output_dir / "matched_drug_response_pairs.csv",
    index=False
)

print("\nSaved: matched_drug_response_pairs.csv")

display(response_pairs.head())


=== FINAL OBSERVATION TABLE ===
Total observations: 106153
Unique cell lines: 491
Unique drugs: 266

Responses by sex:
           observations  cell_lines
final_sex                          
Female            47726         224
Male              58427         267

Duplicate cell-drug pairs: 0

Missing responses: 0

Saved: matched_drug_response_pairs.csv


,SIDM,ModelID,Drug_ID,response,final_sex,label_confidence,Tissue_type
0,SIDM00312,ACH-000183,GDSC:1,2.812192,Female,low,Haematopoietic and Lymphoid
1,SIDM00554,ACH-000524,GDSC:1,2.618159,Female,low,Ovary
2,SIDM00142,ACH-000864,GDSC:1,3.641690,Female,high,Endometrium
3,SIDM01129,ACH-000816,GDSC:1,2.899235,Male,low,Lung
4,SIDM01077,ACH-000115,GDSC:1,3.338584,Male,high,Prostate


In [10]:

import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

# One row per eligible cell line
cells_for_split = eligible[
    ["SIDM", "final_sex", "Tissue_type"]
].drop_duplicates("SIDM").copy()

# First: 70% train, 30% remaining
train_cells, remaining_cells = train_test_split(
    cells_for_split,
    test_size=0.30,
    random_state=42,
    stratify=cells_for_split["final_sex"]
)

# Second: divide remaining cells equally
val_cells, test_cells = train_test_split(
    remaining_cells,
    test_size=0.50,
    random_state=42,
    stratify=remaining_cells["final_sex"]
)

# Create sample-to-split lookup
split_lookup = {}

for label, df in [
    ("train", train_cells),
    ("val", val_cells),
    ("test", test_cells)
]:
    for sidm in df["SIDM"]:
        split_lookup[sidm] = label

# Assign each drug-response pair
response_pairs["split"] = (
    response_pairs["SIDM"].map(split_lookup)
)

# Verify that no cell line occurs in multiple splits
cell_overlap = (
    response_pairs.groupby("SIDM")["split"].nunique()
)

assert (cell_overlap == 1).all(), (
    "Cell-line leakage detected!"
)

assert response_pairs["split"].notna().all()

print("=== TEMPORARY CELL-BLIND SPLIT ===")

print("\nCell lines per split:")
print(
    response_pairs.groupby("split")["SIDM"].nunique()
)

print("\nDrug-response pairs per split:")
print(response_pairs["split"].value_counts())

print("\nSex distribution by split:")
print(
    pd.crosstab(
        cells_for_split["SIDM"].map(split_lookup),
        cells_for_split["final_sex"]
    )
)

print("\nCell-line leakage: NONE")

# Save temporary split
output_dir = Path("/content/week3_outputs")

response_pairs.to_csv(
    output_dir / "response_pairs_temporary_split.csv",
    index=False
)

print("\nSaved: response_pairs_temporary_split.csv")


=== TEMPORARY CELL-BLIND SPLIT ===

Cell lines per split:
split
test      74
train    343
val       74
Name: SIDM, dtype: int64

Drug-response pairs per split:
split
train    74116
val      16072
test     15965
Name: count, dtype: int64

Sex distribution by split:
final_sex  Female  Male
SIDM                   
test           34    40
train         156   187
val            34    40

Cell-line leakage: NONE

Saved: response_pairs_temporary_split.csv


In [11]:

import torch
import torch.nn as nn

class OmicsAttentionFusion(nn.Module):

    def __init__(
        self,
        input_dims,
        n_drugs,
        hidden_dim=64,
        use_proteomics=True,
        use_sex=False
    ):
        super().__init__()

        self.use_proteomics = use_proteomics
        self.use_sex = use_sex

        self.modalities = [
            "expression",
            "mutation",
            "cnv"
        ]

        if use_proteomics:
            self.modalities.append("proteomics")

        # Separate encoder for each omics layer
        self.encoders = nn.ModuleDict({
            name: nn.Sequential(
                nn.Linear(input_dims[name], 128),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(128, hidden_dim)
            )
            for name in self.modalities
        })

        # Learn which modalities interact
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True
        )

        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=2,
            enable_nested_tensor=False
        )

        # Learned drug representation
        self.drug_embedding = nn.Embedding(
            n_drugs,
            hidden_dim
        )

        # Optional sex representation
        self.sex_embedding = (
            nn.Embedding(2, hidden_dim)
            if use_sex else None
        )

        # Final prediction head
        prediction_dim = hidden_dim * 2

        if use_sex:
            prediction_dim += hidden_dim

        self.predictor = nn.Sequential(
            nn.Linear(prediction_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, omics, drug_ids, sex_ids=None):

        # Encode each available omics layer
        tokens = [
            self.encoders[name](omics[name])
            for name in self.modalities
        ]

        # Shape: batch x modalities x hidden_dim
        tokens = torch.stack(tokens, dim=1)

        # Attention across omics layers
        fused = self.transformer(tokens).mean(dim=1)

        # Drug representation
        drug_vector = self.drug_embedding(drug_ids)

        features = [fused, drug_vector]

        # Optional sex-aware component
        if self.use_sex:
            if sex_ids is None:
                raise ValueError(
                    "sex_ids required when use_sex=True"
                )

            sex_vector = self.sex_embedding(sex_ids)
            features.append(sex_vector)

        combined = torch.cat(features, dim=1)

        return self.predictor(combined).squeeze(-1)


print("Attention-fusion model defined successfully!")


Attention-fusion model defined successfully!


In [12]:

from pathlib import Path
import sys
import numpy as np
import pandas as pd

# Your Week 3 datasets from Step 3
omics_matrices = {
    "expression": expression,
    "mutation": mutation,
    "cnv": cnv,
    "proteomics": proteomics
}

# Use the cell-line identifiers from Step 10
train_ids = train_cells["SIDM"].astype(str).tolist()
eligible_ids = eligible["SIDM"].astype(str).tolist()

# Verify the samples exist in all four matrices
for name, df in omics_matrices.items():
    missing_ids = set(eligible_ids) - set(df.index.astype(str))
    assert not missing_ids, (
        f"{name} is missing {len(missing_ids)} cell lines"
    )

print("Training cell lines:", len(train_ids))
print("Eligible cell lines:", len(eligible_ids))

# Week 2 pipeline script must be available in this
# Colab session (upload the .py file if necessary).
pipeline_path = Path(
    "/content/multiomics_preprocessing_pipeline.py"
)

if not pipeline_path.exists():
    print("\nACTION NEEDED:")
    print(
        "Upload multiomics_preprocessing_pipeline.py "
        "from your Week 2 work into Colab."
    )
else:
    from multiomics_preprocessing_pipeline import (
        MultiOmicsPreprocessor
    )

    processor = MultiOmicsPreprocessor(
        protein_min_coverage=0.70
    )

    processor.fit(omics_matrices, train_ids)

    processed_omics = processor.transform(
        omics_matrices,
        eligible_ids
    )

    print("\n=== PREPROCESSED OMICS ===")

    for name, df in processed_omics.items():
        print(
            name,
            "| Shape:", df.shape,
            "| All finite:",
            np.isfinite(df.to_numpy()).all()
        )

    print("\nPreprocessing complete!")


Training cell lines: 343
Eligible cell lines: 491

=== PREPROCESSED OMICS ===
expression | Shape: (491, 17990) | All finite: True
mutation | Shape: (491, 1650) | All finite: True
cnv | Shape: (491, 23264) | All finite: True
proteomics | Shape: (491, 4127) | All finite: True

Preprocessing complete!


In [13]:

import numpy as np
import pandas as pd
import torch

# Choose CPU or GPU
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

# Keep cell lines in one consistent order
cell_ids = eligible["SIDM"].astype(str).tolist()

# Create cell-line index mapping
cell_to_idx = {
    sidm: i for i, sidm in enumerate(cell_ids)
}

# Create drug index mapping
drug_ids = sorted(
    response_pairs["Drug_ID"].unique()
)

drug_to_idx = {
    drug: i for i, drug in enumerate(drug_ids)
}

# Create PyTorch molecular feature tensors
omics_tensors = {}

for name, df in processed_omics.items():

    aligned = df.loc[cell_ids]

    values = aligned.to_numpy(
        dtype=np.float32
    )

    assert np.isfinite(values).all()

    omics_tensors[name] = torch.tensor(
        values,
        dtype=torch.float32
    )

# Add numerical IDs to the response table
model_pairs = response_pairs.copy()

model_pairs["cell_idx"] = (
    model_pairs["SIDM"].map(cell_to_idx)
)

model_pairs["drug_idx"] = (
    model_pairs["Drug_ID"].map(drug_to_idx)
)

# Encode sex: Female = 0, Male = 1
model_pairs["sex_idx"] = (
    model_pairs["final_sex"].map({
        "Female": 0,
        "Male": 1
    })
)

assert model_pairs[
    ["cell_idx", "drug_idx", "sex_idx"]
].notna().all().all()

# Convert to integer indices
for col in ["cell_idx", "drug_idx", "sex_idx"]:
    model_pairs[col] = model_pairs[col].astype(int)

# Build model input dimensions
input_dims = {
    name: tensor.shape[1]
    for name, tensor in omics_tensors.items()
}

print("\n=== MODEL INPUT DIMENSIONS ===")
print(input_dims)

print("\nTotal cell lines:", len(cell_ids))
print("Total drugs:", len(drug_ids))
print("Total observations:", len(model_pairs))

print("\nFirst five indexed observations:")
display(
    model_pairs[
        [
            "SIDM",
            "Drug_ID",
            "cell_idx",
            "drug_idx",
            "sex_idx",
            "response",
            "split"
        ]
    ].head()
)


Device: cuda

=== MODEL INPUT DIMENSIONS ===
{'expression': 17990, 'mutation': 1650, 'cnv': 23264, 'proteomics': 4127}

Total cell lines: 491
Total drugs: 266
Total observations: 106153

First five indexed observations:


,SIDM,Drug_ID,cell_idx,drug_idx,sex_idx,response,split
0,SIDM00312,GDSC:1,0,0,0,2.812192,train
1,SIDM00554,GDSC:1,5,0,0,2.618159,train
2,SIDM00142,GDSC:1,7,0,0,3.641690,train
3,SIDM01129,GDSC:1,8,0,1,2.899235,val
4,SIDM01077,GDSC:1,13,0,1,3.338584,train


In [14]:

import torch

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


Device: cuda
GPU: Tesla T4


In [15]:

import torch

# Select a small batch of real observations
batch = model_pairs.head(4)

cell_idx = torch.tensor(
    batch["cell_idx"].values,
    dtype=torch.long,
    device=device
)

drug_idx = torch.tensor(
    batch["drug_idx"].values,
    dtype=torch.long,
    device=device
)

sex_idx = torch.tensor(
    batch["sex_idx"].values,
    dtype=torch.long,
    device=device
)

# Four combinations required for Week 3
configurations = {
    "A_baseline": (False, False),
    "B_proteomics": (True, False),
    "C_sex_aware": (False, True),
    "D_full": (True, True)
}

for name, (use_proteomics, use_sex) in configurations.items():

    model = OmicsAttentionFusion(
        input_dims=input_dims,
        n_drugs=len(drug_ids),
        use_proteomics=use_proteomics,
        use_sex=use_sex
    ).to(device)

    batch_omics = {
        key: tensor[cell_idx.cpu()].to(device)
        for key, tensor in omics_tensors.items()
    }

    model.eval()

    with torch.no_grad():
        predictions = model(
            batch_omics,
            drug_idx,
            sex_idx if use_sex else None
        )

    print(f"\n{name}")
    print("Predictions shape:", tuple(predictions.shape))
    print(
        "All finite:",
        bool(torch.isfinite(predictions).all())
    )
    print("Predictions:", predictions.cpu().numpy().round(4))

    del model

print("\nAll four model configurations tested!")



A_baseline
Predictions shape: (4,)
All finite: True
Predictions: [0.0761 0.0977 0.1148 0.0781]

B_proteomics
Predictions shape: (4,)
All finite: True
Predictions: [0.0311 0.0624 0.0302 0.0502]

C_sex_aware
Predictions shape: (4,)
All finite: True
Predictions: [0.106  0.1543 0.1301 0.1838]

D_full
Predictions shape: (4,)
All finite: True
Predictions: [-0.1068 -0.1133 -0.1173 -0.203 ]

All four model configurations tested!


In [16]:

import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np

class DrugResponseDataset(Dataset):
    def __init__(self, dataframe):
        self.cell_idx = torch.tensor(
            dataframe["cell_idx"].to_numpy(),
            dtype=torch.long
        )
        self.drug_idx = torch.tensor(
            dataframe["drug_idx"].to_numpy(),
            dtype=torch.long
        )
        self.sex_idx = torch.tensor(
            dataframe["sex_idx"].to_numpy(),
            dtype=torch.long
        )
        self.targets = torch.tensor(
            dataframe["response"].to_numpy(dtype=np.float32),
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return (
            self.cell_idx[idx],
            self.drug_idx[idx],
            self.sex_idx[idx],
            self.targets[idx]
        )

train_dataset = DrugResponseDataset(
    model_pairs[model_pairs["split"] == "train"]
)

val_dataset = DrugResponseDataset(
    model_pairs[model_pairs["split"] == "val"]
)

test_dataset = DrugResponseDataset(
    model_pairs[model_pairs["split"] == "test"]
)

batch_size = 256

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)

print("=== TRAINING DATA READY ===")
print("Train observations:", len(train_dataset))
print("Validation observations:", len(val_dataset))
print("Test observations:", len(test_dataset))

# Verify one batch
cell_batch, drug_batch, sex_batch, y_batch = next(
    iter(train_loader)
)

print("\nSample batch shapes:")
print("Cell indices:", cell_batch.shape)
print("Drug indices:", drug_batch.shape)
print("Sex indices:", sex_batch.shape)
print("Targets:", y_batch.shape)

print("\nFirst five target values:")
print(y_batch[:5].numpy())


=== TRAINING DATA READY ===
Train observations: 74116
Validation observations: 16072
Test observations: 15965

Sample batch shapes:
Cell indices: torch.Size([256])
Drug indices: torch.Size([256])
Sex indices: torch.Size([256])
Targets: torch.Size([256])

First five target values:
[1.206434 4.98326  4.752476 6.547528 3.989669]


In [17]:

import torch
import torch.nn as nn
import numpy as np
import copy
import random

from pathlib import Path

# Keep experiments reproducible
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Move the 491 molecular feature vectors to the device.
# They are indexed by cell line rather than copied per pair.
omics_device = {
    name: tensor.to(device)
    for name, tensor in omics_tensors.items()
}

def train_fusion_model(
    model,
    train_loader,
    val_loader,
    epochs=15,
    learning_rate=0.001,
    patience=4
):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )

    criterion = nn.MSELoss()

    best_val_rmse = float("inf")
    best_state = None
    patience_counter = 0

    history = []

    for epoch in range(epochs):

        # Training
        model.train()
        train_squared_error = 0.0
        train_count = 0

        for cells, drugs, sexes, targets in train_loader:
            cells = cells.to(device)
            drugs = drugs.to(device)
            sexes = sexes.to(device)
            targets = targets.to(device)

            batch_omics = {
                name: values[cells]
                for name, values in omics_device.items()
                if name in model.modalities
            }

            optimizer.zero_grad()

            predictions = model(
                batch_omics,
                drugs,
                sexes if model.use_sex else None
            )

            loss = criterion(predictions, targets)

            loss.backward()

            optimizer.step()

            train_squared_error += (
                (predictions.detach() - targets)
                .square().sum().item()
            )

            train_count += len(targets)

        train_rmse = np.sqrt(
            train_squared_error / train_count
        )

        # Validation
        model.eval()
        val_squared_error = 0.0
        val_count = 0

        with torch.no_grad():

            for cells, drugs, sexes, targets in val_loader:
                cells = cells.to(device)
                drugs = drugs.to(device)
                sexes = sexes.to(device)
                targets = targets.to(device)

                batch_omics = {
                    name: values[cells]
                    for name, values in omics_device.items()
                    if name in model.modalities
                }

                predictions = model(
                    batch_omics,
                    drugs,
                    sexes if model.use_sex else None
                )

                val_squared_error += (
                    (predictions - targets)
                    .square().sum().item()
                )

                val_count += len(targets)

        val_rmse = np.sqrt(
            val_squared_error / val_count
        )

        history.append({
            "epoch": epoch + 1,
            "train_rmse": train_rmse,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train RMSE: {train_rmse:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        # Save the best validation checkpoint
        if val_rmse < best_val_rmse - 1e-4:
            best_val_rmse = val_rmse
            best_state = copy.deepcopy(
                model.state_dict()
            )
            patience_counter = 0

        else:
            patience_counter += 1

            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    # Restore best checkpoint
    model.load_state_dict(best_state)

    return model, history, best_val_rmse


print("Training function defined successfully!")
print("Device:", device)


Training function defined successfully!
Device: cuda


In [18]:

import pandas as pd

split_file = "/content/Train_Val_Freeze Splits - Celusaurous .xlsx"

# Check workbook structure
xls = pd.ExcelFile(split_file)

print("Available sheets:", xls.sheet_names)

for sheet in xls.sheet_names:
    df = pd.read_excel(split_file, sheet_name=sheet)

    print(f"\n=== {sheet} ===")
    print("Columns:", df.columns.tolist())
    print("Total rows:", len(df))

    display(df.head(3))


Available sheets: ['cell line splits', 'tissue splits']

=== cell line splits ===
Columns: ['sample_id', 'cell_line_id', 'tissue', 'split']
Total rows: 526


,sample_id,cell_line_id,tissue,split
0,SIDM00081,SIDM00081,Skin,test
1,SIDM00140,SIDM00140,Skin,test
2,SIDM00143,SIDM00143,Lung,test



=== tissue splits ===
Columns: ['sample_id', 'cell_line_id', 'tissue', 'split']
Total rows: 526


,sample_id,cell_line_id,tissue,split
0,SIDM00080,SIDM00080,Skin,test
1,SIDM00081,SIDM00081,Skin,test
2,SIDM00082,SIDM00082,Skin,test


In [19]:

# Your 491 eligible cell lines
eligible_ids = set(eligible["SIDM"].astype(str))

# Load two official splits
cell_splits = pd.read_excel(
    split_file,
    sheet_name="cell line splits"
)

tissue_splits = pd.read_excel(
    split_file,
    sheet_name="tissue splits"
)

for name, df in [
    ("CELL-LINE-BLIND", cell_splits),
    ("TISSUE-BLIND", tissue_splits)
]:
    print(f"\n=== {name} ===")

    # Check for duplicate cell-line IDs
    assert not df["cell_line_id"].duplicated().any(), (
        f"Duplicate cell lines found in {name}"
    )

    matched = df[
        df["cell_line_id"].astype(str).isin(eligible_ids)
    ].copy()

    matched_ids = set(
        matched["cell_line_id"].astype(str)
    )

    print("Eligible cell lines matched:", len(matched))
    print("\nSplit distribution:")
    print(matched["split"].value_counts())

    print("\nMissing eligible cell lines:",
          len(eligible_ids - matched_ids))

    print("Complete coverage:",
          matched_ids == eligible_ids)

    # Check for unexpected split labels
    print("Split labels:", sorted(df["split"].unique()))

    if name == "TISSUE-BLIND":
        tissue_overlap = matched.groupby("tissue")["split"].nunique()
        print(
            "Tissues appearing in multiple splits:",
            int((tissue_overlap > 1).sum())
        )



=== CELL-LINE-BLIND ===
Eligible cell lines matched: 491

Split distribution:
split
train    345
val       74
test      72
Name: count, dtype: int64

Missing eligible cell lines: 0
Complete coverage: True
Split labels: ['test', 'train', 'val']

=== TISSUE-BLIND ===
Eligible cell lines matched: 491

Split distribution:
split
train    338
test      79
val       74
Name: count, dtype: int64

Missing eligible cell lines: 0
Complete coverage: True
Split labels: ['test', 'train', 'val']
Tissues appearing in multiple splits: 0


In [20]:

# STEP 29: Apply official cell-line-blind splits

official_split_lookup = dict(
    zip(
        cell_splits["cell_line_id"].astype(str),
        cell_splits["split"].astype(str)
    )
)

# Assign official splits to eligible cell lines
eligible["official_split"] = (
    eligible["SIDM"]
    .astype(str)
    .map(official_split_lookup)
)

assert eligible["official_split"].notna().all()

# Replace temporary split assignments
response_pairs["split"] = (
    response_pairs["SIDM"]
    .astype(str)
    .map(official_split_lookup)
)

assert response_pairs["split"].notna().all()

# Recreate training/validation/test cell-line groups
train_cells = eligible[
    eligible["official_split"] == "train"
].copy()

val_cells = eligible[
    eligible["official_split"] == "val"
].copy()

test_cells = eligible[
    eligible["official_split"] == "test"
].copy()

print("=== OFFICIAL CELL-LINE SPLITS ===")
print("Train cells:", len(train_cells))
print("Val cells:", len(val_cells))
print("Test cells:", len(test_cells))

print("\n=== DRUG-RESPONSE OBSERVATIONS ===")
print(response_pairs["split"].value_counts())

print("\n=== SEX DISTRIBUTION ===")
print(
    eligible.groupby(
        ["official_split", "final_sex"]
    ).size()
)

# Save the official split assignments
response_pairs.to_csv(
    "/content/week3_outputs/response_pairs_official_cell_split.csv",
    index=False
)

print("\nOfficial split assignments saved!")


=== OFFICIAL CELL-LINE SPLITS ===
Train cells: 345
Val cells: 74
Test cells: 72

=== DRUG-RESPONSE OBSERVATIONS ===
split
train    75028
val      15624
test     15501
Name: count, dtype: int64

=== SEX DISTRIBUTION ===
official_split  final_sex
test            Female        33
                Male          39
train           Female       153
                Male         192
val             Female        38
                Male          36
dtype: int64

Official split assignments saved!


In [21]:

# STEP 30: Refit preprocessing on official training cells

import numpy as np
import pandas as pd
from multiomics_preprocessing_pipeline import MultiOmicsPreprocessor

# Recreate the original molecular matrices
omics_matrices = {
    "expression": expression,
    "mutation": mutation,
    "cnv": cnv,
    "proteomics": proteomics
}

# Official training IDs
official_train_ids = (
    train_cells["SIDM"].astype(str).tolist()
)

# All 491 eligible cell lines
eligible_ids = (
    eligible["SIDM"].astype(str).tolist()
)

# Create a NEW processor for the official split
processor = MultiOmicsPreprocessor(
    protein_min_coverage=0.70
)

# Fit ONLY on official training cells
processor.fit(
    omics_matrices,
    official_train_ids
)

# Transform all eligible cells using training-fitted parameters
processed_omics = processor.transform(
    omics_matrices,
    eligible_ids
)

print("=== OFFICIAL PREPROCESSING COMPLETE ===")

for name, matrix in processed_omics.items():

    values = matrix.to_numpy(dtype=np.float32)

    print(
        f"{name}: {values.shape} | "
        f"All finite: {np.isfinite(values).all()}"
    )

print("\nPreprocessing fitted on:",
      len(official_train_ids),
      "official training cells")

print("Total transformed cells:",
      len(eligible_ids))


=== OFFICIAL PREPROCESSING COMPLETE ===
expression: (491, 17996) | All finite: True
mutation: (491, 1648) | All finite: True
cnv: (491, 23264) | All finite: True
proteomics: (491, 4153) | All finite: True

Preprocessing fitted on: 345 official training cells
Total transformed cells: 491


In [22]:

# STEP 31: Rebuild inputs for official cell-line-blind split

import torch
import numpy as np
from torch.utils.data import DataLoader

# Preserve the same cell-line ordering
cell_ids = eligible["SIDM"].astype(str).tolist()
cell_to_idx = {
    sidm: i for i, sidm in enumerate(cell_ids)
}

# Recreate molecular tensors
omics_tensors = {
    name: torch.tensor(
        processed_omics[name]
        .loc[cell_ids]
        .to_numpy(dtype=np.float32)
    )
    for name in processed_omics
}

# Recalculate feature dimensions
input_dims = {
    name: tensor.shape[1]
    for name, tensor in omics_tensors.items()
}

# Rebuild response-pair mapping
model_pairs = response_pairs.copy()

model_pairs["cell_idx"] = (
    model_pairs["SIDM"].astype(str).map(cell_to_idx)
)

# Preserve the original drug mapping
model_pairs["drug_idx"] = (
    model_pairs["Drug_ID"].map(drug_to_idx)
)

model_pairs["sex_idx"] = (
    model_pairs["final_sex"].map({
        "Female": 0,
        "Male": 1
    })
)

# Validate mappings
assert model_pairs[
    ["cell_idx", "drug_idx", "sex_idx"]
].notna().all().all()

# Ensure official splits are present
assert set(model_pairs["split"].unique()) == {
    "train", "val", "test"
}

# Recreate datasets using the existing Dataset class
train_dataset = DrugResponseDataset(
    model_pairs[model_pairs["split"] == "train"]
)

val_dataset = DrugResponseDataset(
    model_pairs[model_pairs["split"] == "val"]
)

test_dataset = DrugResponseDataset(
    model_pairs[model_pairs["split"] == "test"]
)

# Recreate loaders
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=256,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False
)

# IMPORTANT: Refresh device-side molecular tensors
omics_device = {
    name: tensor.to(device)
    for name, tensor in omics_tensors.items()
}

print("=== OFFICIAL TRAINING INPUTS READY ===")

print("\nFeature dimensions:")
print(input_dims)

print("\nDrug-response observations:")
print("Training:", len(train_dataset))
print("Validation:", len(val_dataset))
print("Testing:", len(test_dataset))

print("\nCell lines:")
print("Training:", len(train_cells))
print("Validation:", len(val_cells))
print("Testing:", len(test_cells))

print("\nDevice:", device)
print("All feature tensors updated:", all(
    omics_device[name].shape[1] == input_dims[name]
    for name in input_dims
))


=== OFFICIAL TRAINING INPUTS READY ===

Feature dimensions:
{'expression': 17996, 'mutation': 1648, 'cnv': 23264, 'proteomics': 4153}

Drug-response observations:
Training: 75028
Validation: 15624
Testing: 15501

Cell lines:
Training: 345
Validation: 74
Testing: 72

Device: cuda
All feature tensors updated: True


In [23]:

# STEP 32: Train Model A using official cell-line-blind split

import torch
import pandas as pd
from pathlib import Path

# Separate folder for official experiments
official_dir = Path(
    "/content/week3_outputs/official_cell_blind"
)
official_dir.mkdir(parents=True, exist_ok=True)

# Reproducibility
set_seed(42)

# Initialize a fresh baseline model
official_model_A = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=False,
    use_sex=False
)

print("=== OFFICIAL MODEL A ===")
print("Modalities:", official_model_A.modalities)
print("Sex-aware:", official_model_A.use_sex)
print("Training observations:", len(train_dataset))
print("Validation observations:", len(val_dataset))

# Train on official split
official_model_A, official_history_A, official_val_A = (
    train_fusion_model(
        model=official_model_A,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save trained checkpoint
torch.save(
    official_model_A.state_dict(),
    official_dir / "model_A_baseline.pt"
)

# Save training history
pd.DataFrame(official_history_A).to_csv(
    official_dir / "model_A_training_history.csv",
    index=False
)

print("\n=== OFFICIAL MODEL A COMPLETE ===")
print(
    "Best validation RMSE:",
    round(official_val_A, 4)
)
print("Checkpoint saved successfully!")


=== OFFICIAL MODEL A ===
Modalities: ['expression', 'mutation', 'cnv']
Sex-aware: False
Training observations: 75028
Validation observations: 15624
Epoch 01 | Train RMSE: 1.7774 | Val RMSE: 1.3954
Epoch 02 | Train RMSE: 1.2959 | Val RMSE: 1.3452
Epoch 03 | Train RMSE: 1.2526 | Val RMSE: 1.3281
Epoch 04 | Train RMSE: 1.2214 | Val RMSE: 1.3200
Epoch 05 | Train RMSE: 1.2108 | Val RMSE: 1.3116
Epoch 06 | Train RMSE: 1.1938 | Val RMSE: 1.3125
Epoch 07 | Train RMSE: 1.1835 | Val RMSE: 1.3171
Epoch 08 | Train RMSE: 1.1740 | Val RMSE: 1.3446
Epoch 09 | Train RMSE: 1.1654 | Val RMSE: 1.3264
Early stopping triggered.

=== OFFICIAL MODEL A COMPLETE ===
Best validation RMSE: 1.3116
Checkpoint saved successfully!


In [24]:

# STEP 33: Train official Model B with proteomics

import torch
import pandas as pd

set_seed(42)

official_model_B = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=True,
    use_sex=False
)

print("=== OFFICIAL MODEL B ===")
print("Modalities:", official_model_B.modalities)
print("Sex-aware:", official_model_B.use_sex)

official_model_B, official_history_B, official_val_B = (
    train_fusion_model(
        model=official_model_B,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save official checkpoint and history
torch.save(
    official_model_B.state_dict(),
    official_dir / "model_B_proteomics.pt"
)

pd.DataFrame(official_history_B).to_csv(
    official_dir / "model_B_training_history.csv",
    index=False
)

print("\n=== OFFICIAL VALIDATION COMPARISON ===")
print("Model A RMSE:", round(official_val_A, 4))
print("Model B RMSE:", round(official_val_B, 4))

difference = official_val_B - official_val_A

print("Difference (B - A):", round(difference, 4))

if difference < 0:
    print("Proteomics improved validation RMSE.")
elif difference > 0:
    print("Proteomics worsened validation RMSE.")
else:
    print("Validation RMSE was unchanged.")

print("\nOfficial Model B checkpoint saved!")


=== OFFICIAL MODEL B ===
Modalities: ['expression', 'mutation', 'cnv', 'proteomics']
Sex-aware: False
Epoch 01 | Train RMSE: 1.8468 | Val RMSE: 1.3858
Epoch 02 | Train RMSE: 1.3040 | Val RMSE: 1.3194
Epoch 03 | Train RMSE: 1.2523 | Val RMSE: 1.3464
Epoch 04 | Train RMSE: 1.2306 | Val RMSE: 1.3242
Epoch 05 | Train RMSE: 1.2076 | Val RMSE: 1.3209
Epoch 06 | Train RMSE: 1.1945 | Val RMSE: 1.3192
Epoch 07 | Train RMSE: 1.1833 | Val RMSE: 1.3392
Epoch 08 | Train RMSE: 1.1768 | Val RMSE: 1.3108
Epoch 09 | Train RMSE: 1.1604 | Val RMSE: 1.3176
Epoch 10 | Train RMSE: 1.1501 | Val RMSE: 1.3208
Epoch 11 | Train RMSE: 1.1399 | Val RMSE: 1.3168
Epoch 12 | Train RMSE: 1.1331 | Val RMSE: 1.3113
Early stopping triggered.

=== OFFICIAL VALIDATION COMPARISON ===
Model A RMSE: 1.3116
Model B RMSE: 1.3108
Difference (B - A): -0.0008
Proteomics improved validation RMSE.

Official Model B checkpoint saved!


In [25]:

# STEP 34: Train official Model C with sex-aware input

import torch
import pandas as pd

set_seed(42)

official_model_C = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=False,
    use_sex=True
)

print("=== OFFICIAL MODEL C ===")
print("Modalities:", official_model_C.modalities)
print("Sex-aware:", official_model_C.use_sex)

official_model_C, official_history_C, official_val_C = (
    train_fusion_model(
        model=official_model_C,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save official checkpoint and history
torch.save(
    official_model_C.state_dict(),
    official_dir / "model_C_sex_aware.pt"
)

pd.DataFrame(official_history_C).to_csv(
    official_dir / "model_C_training_history.csv",
    index=False
)

print("\n=== OFFICIAL VALIDATION COMPARISON ===")
print("Model A RMSE:", round(official_val_A, 4))
print("Model C RMSE:", round(official_val_C, 4))

difference = official_val_C - official_val_A

print("Difference (C - A):", round(difference, 4))

if difference < 0:
    print("Sex-aware input improved validation RMSE.")
elif difference > 0:
    print("Sex-aware input worsened validation RMSE.")
else:
    print("Validation RMSE was unchanged.")

print("\nOfficial Model C checkpoint saved!")


=== OFFICIAL MODEL C ===
Modalities: ['expression', 'mutation', 'cnv']
Sex-aware: True
Epoch 01 | Train RMSE: 1.7909 | Val RMSE: 1.3741
Epoch 02 | Train RMSE: 1.2910 | Val RMSE: 1.4003
Epoch 03 | Train RMSE: 1.2519 | Val RMSE: 1.3323
Epoch 04 | Train RMSE: 1.2275 | Val RMSE: 1.3319
Epoch 05 | Train RMSE: 1.2105 | Val RMSE: 1.3186
Epoch 06 | Train RMSE: 1.1936 | Val RMSE: 1.3282
Epoch 07 | Train RMSE: 1.1801 | Val RMSE: 1.3241
Epoch 08 | Train RMSE: 1.1762 | Val RMSE: 1.3188
Epoch 09 | Train RMSE: 1.1636 | Val RMSE: 1.3154
Epoch 10 | Train RMSE: 1.1553 | Val RMSE: 1.3421
Epoch 11 | Train RMSE: 1.1458 | Val RMSE: 1.3229
Epoch 12 | Train RMSE: 1.1464 | Val RMSE: 1.3242
Epoch 13 | Train RMSE: 1.1384 | Val RMSE: 1.3093
Epoch 14 | Train RMSE: 1.1270 | Val RMSE: 1.3087
Epoch 15 | Train RMSE: 1.1233 | Val RMSE: 1.3235

=== OFFICIAL VALIDATION COMPARISON ===
Model A RMSE: 1.3116
Model C RMSE: 1.3087
Difference (C - A): -0.0029
Sex-aware input improved validation RMSE.

Official Model C checkpoi

In [26]:

# STEP 35: Train official Model D (full fusion)

import torch
import pandas as pd

set_seed(42)

official_model_D = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=True,
    use_sex=True
)

print("=== OFFICIAL MODEL D ===")
print("Modalities:", official_model_D.modalities)
print("Sex-aware:", official_model_D.use_sex)

official_model_D, official_history_D, official_val_D = (
    train_fusion_model(
        model=official_model_D,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save official checkpoint and history
torch.save(
    official_model_D.state_dict(),
    official_dir / "model_D_full_fusion.pt"
)

pd.DataFrame(official_history_D).to_csv(
    official_dir / "model_D_training_history.csv",
    index=False
)

print("\n=== OFFICIAL FOUR-MODEL VALIDATION COMPARISON ===")

official_results = pd.DataFrame([
    {
        "Model": "A_baseline",
        "Proteomics": False,
        "Sex_aware": False,
        "Best_val_RMSE": official_val_A
    },
    {
        "Model": "B_proteomics",
        "Proteomics": True,
        "Sex_aware": False,
        "Best_val_RMSE": official_val_B
    },
    {
        "Model": "C_sex_aware",
        "Proteomics": False,
        "Sex_aware": True,
        "Best_val_RMSE": official_val_C
    },
    {
        "Model": "D_full",
        "Proteomics": True,
        "Sex_aware": True,
        "Best_val_RMSE": official_val_D
    }
])

official_results = official_results.sort_values(
    "Best_val_RMSE"
)

display(official_results.round(4))

official_results.to_csv(
    official_dir / "four_model_validation_comparison.csv",
    index=False
)

print("\nOfficial Model D checkpoint saved!")
print("Official validation comparison saved!")


=== OFFICIAL MODEL D ===
Modalities: ['expression', 'mutation', 'cnv', 'proteomics']
Sex-aware: True
Epoch 01 | Train RMSE: 1.8561 | Val RMSE: 1.3837
Epoch 02 | Train RMSE: 1.3083 | Val RMSE: 1.3485
Epoch 03 | Train RMSE: 1.2615 | Val RMSE: 1.3307
Epoch 04 | Train RMSE: 1.2267 | Val RMSE: 1.3311
Epoch 05 | Train RMSE: 1.2107 | Val RMSE: 1.3025
Epoch 06 | Train RMSE: 1.1941 | Val RMSE: 1.3161
Epoch 07 | Train RMSE: 1.1806 | Val RMSE: 1.3244
Epoch 08 | Train RMSE: 1.1778 | Val RMSE: 1.3141
Epoch 09 | Train RMSE: 1.1618 | Val RMSE: 1.3011
Epoch 10 | Train RMSE: 1.1525 | Val RMSE: 1.3084
Epoch 11 | Train RMSE: 1.1460 | Val RMSE: 1.3120
Epoch 12 | Train RMSE: 1.1318 | Val RMSE: 1.3135
Epoch 13 | Train RMSE: 1.1259 | Val RMSE: 1.3109
Early stopping triggered.

=== OFFICIAL FOUR-MODEL VALIDATION COMPARISON ===


,Model,Proteomics,Sex_aware,Best_val_RMSE
3,D_full,True,True,1.3011
2,C_sex_aware,False,True,1.3087
1,B_proteomics,True,False,1.3108
0,A_baseline,False,False,1.3116



Official Model D checkpoint saved!
Official validation comparison saved!


In [28]:

# STEP 36 FIX: Restore the evaluation function

import numpy as np
import pandas as pd
import torch
from scipy.stats import pearsonr

def evaluate_model_by_sex(model, loader, model_name):

    model = model.to(device)
    model.eval()

    predictions = []
    targets_all = []
    sex_all = []

    with torch.no_grad():

        for cells, drugs, sexes, targets in loader:

            cells = cells.to(device)
            drugs = drugs.to(device)
            sexes = sexes.to(device)

            batch_omics = {
                name: values[cells]
                for name, values in omics_device.items()
                if name in model.modalities
            }

            pred = model(
                batch_omics,
                drugs,
                sexes if model.use_sex else None
            )

            predictions.extend(
                pred.cpu().numpy()
            )

            targets_all.extend(
                targets.numpy()
            )

            sex_all.extend(
                sexes.cpu().numpy()
            )

    df = pd.DataFrame({
        "actual": targets_all,
        "predicted": predictions,
        "sex_idx": sex_all
    })

    df["sex"] = df["sex_idx"].map({
        0: "Female",
        1: "Male"
    })

    results = []

    for group, subset in [
        ("Overall", df),
        ("Female", df[df["sex"] == "Female"]),
        ("Male", df[df["sex"] == "Male"])
    ]:

        y = subset["actual"].to_numpy()
        p = subset["predicted"].to_numpy()

        rmse = np.sqrt(
            np.mean((y - p) ** 2)
        )

        mae = np.mean(
            np.abs(y - p)
        )

        if (
            len(y) > 2
            and np.std(y) > 0
            and np.std(p) > 0
        ):
            corr = pearsonr(y, p).statistic
        else:
            corr = np.nan

        results.append({
            "Model": model_name,
            "Group": group,
            "N_pairs": len(subset),
            "RMSE": float(rmse),
            "MAE": float(mae),
            "Pearson": float(corr)
        })

    return pd.DataFrame(results), df


print("Evaluation function restored successfully!")


Evaluation function restored successfully!


In [29]:

# STEP 36: Evaluate official models on official test set

import pandas as pd
from pathlib import Path

official_models = {
    "A_baseline": official_model_A,
    "B_proteomics": official_model_B,
    "C_sex_aware": official_model_C,
    "D_full": official_model_D
}

official_metrics_list = []
official_predictions = {}

for name, model in official_models.items():

    metrics_df, pred_df = evaluate_model_by_sex(
        model,
        test_loader,
        name
    )

    official_metrics_list.append(metrics_df)
    official_predictions[name] = pred_df

# Combine results
official_test_metrics = pd.concat(
    official_metrics_list,
    ignore_index=True
)

print("=== OFFICIAL HELD-OUT TEST PERFORMANCE ===")
display(official_test_metrics.round(4))

# Compare female/male RMSE gaps
official_gap_table = official_test_metrics.pivot(
    index="Model",
    columns="Group",
    values="RMSE"
)

official_gap_table["Female_minus_Male_RMSE"] = (
    official_gap_table["Female"] -
    official_gap_table["Male"]
)

print("\n=== OFFICIAL SEX-STRATIFIED RMSE GAPS ===")
display(official_gap_table.round(4))

# Save outputs
official_test_metrics.to_csv(
    official_dir / "four_model_test_metrics_by_sex.csv",
    index=False
)

official_gap_table.to_csv(
    official_dir / "four_model_test_rmse_gaps.csv"
)

print("\nOfficial test metrics saved!")


=== OFFICIAL HELD-OUT TEST PERFORMANCE ===


,Model,Group,N_pairs,RMSE,MAE,Pearson
0,A_baseline,Overall,15501,1.3519,1.0194,0.8669
1,A_baseline,Female,6791,1.3681,1.0223,0.8665
2,A_baseline,Male,8710,1.3391,1.0171,0.8673
3,B_proteomics,Overall,15501,1.3417,1.0003,0.8694
4,B_proteomics,Female,6791,1.3694,1.0215,0.8665
5,B_proteomics,Male,8710,1.3196,0.9837,0.8717
6,C_sex_aware,Overall,15501,1.3475,1.0028,0.8671
7,C_sex_aware,Female,6791,1.3831,1.0232,0.8632
8,C_sex_aware,Male,8710,1.3192,0.9869,0.8705
9,D_full,Overall,15501,1.3509,1.0116,0.8661



=== OFFICIAL SEX-STRATIFIED RMSE GAPS ===


Group,Female,Male,Overall,Female_minus_Male_RMSE
Model,,,,
A_baseline,1.3681,1.3391,1.3519,0.0290
B_proteomics,1.3694,1.3196,1.3417,0.0498
C_sex_aware,1.3831,1.3192,1.3475,0.0639
D_full,1.3754,1.3316,1.3509,0.0438



Official test metrics saved!


In [30]:

# STEP 37: Bootstrap official sex-stratified RMSE gaps

import numpy as np
import pandas as pd

# Recover official test observations in the same order
official_test_rows = model_pairs[
    model_pairs["split"] == "test"
].reset_index(drop=True)

def bootstrap_official_sex_gap(
    df, n_boot=1000, seed=42
):
    rng = np.random.default_rng(seed)

    groups = {
        sidm: group
        for sidm, group in df.groupby("SIDM")
    }

    female_ids = df.loc[
        df["sex"] == "Female", "SIDM"
    ].unique()

    male_ids = df.loc[
        df["sex"] == "Male", "SIDM"
    ].unique()

    gaps = []

    for _ in range(n_boot):

        female_sample = rng.choice(
            female_ids,
            size=len(female_ids),
            replace=True
        )

        male_sample = rng.choice(
            male_ids,
            size=len(male_ids),
            replace=True
        )

        female_df = pd.concat(
            [groups[s] for s in female_sample],
            ignore_index=True
        )

        male_df = pd.concat(
            [groups[s] for s in male_sample],
            ignore_index=True
        )

        female_rmse = np.sqrt(
            np.mean(
                (female_df["actual"] -
                 female_df["predicted"]) ** 2
            )
        )

        male_rmse = np.sqrt(
            np.mean(
                (male_df["actual"] -
                 male_df["predicted"]) ** 2
            )
        )

        gaps.append(
            female_rmse - male_rmse
        )

    return np.percentile(
        gaps,
        [2.5, 50, 97.5]
    )


bootstrap_results = []

for name, pred_df in official_predictions.items():

    # Confirm predictions match test observations
    assert len(pred_df) == len(official_test_rows)

    assert np.allclose(
        pred_df["actual"].to_numpy(),
        official_test_rows["response"].to_numpy(),
        atol=1e-5
    ), f"Ordering mismatch for {name}"

    combined = pred_df.copy()

    combined["SIDM"] = (
        official_test_rows["SIDM"].to_numpy()
    )

    lower, median, upper = bootstrap_official_sex_gap(
        combined,
        n_boot=1000,
        seed=42
    )

    bootstrap_results.append({
        "Model": name,
        "Median_gap": median,
        "CI_lower": lower,
        "CI_upper": upper,
        "CI_includes_zero": lower <= 0 <= upper
    })

official_bootstrap = pd.DataFrame(
    bootstrap_results
)

print("=== OFFICIAL SEX GAP BOOTSTRAP ===")

display(
    official_bootstrap.round(4)
)

official_bootstrap.to_csv(
    official_dir / "sex_rmse_gap_bootstrap.csv",
    index=False
)

print("\nOfficial bootstrap results saved!")


=== OFFICIAL SEX GAP BOOTSTRAP ===


,Model,Median_gap,CI_lower,CI_upper,CI_includes_zero
0,A_baseline,0.0264,-0.1152,0.1606,True
1,B_proteomics,0.0511,-0.0737,0.1627,True
2,C_sex_aware,0.0608,-0.0709,0.1829,True
3,D_full,0.0437,-0.0901,0.1772,True



Official bootstrap results saved!


In [31]:

# STEP 38: Paired bootstrap for Model B vs Model A

import numpy as np
import pandas as pd

pred_A = official_predictions["A_baseline"]
pred_B = official_predictions["B_proteomics"]

# Verify consistent observation ordering
assert len(pred_A) == len(pred_B) == len(official_test_rows)

assert np.allclose(
    pred_A["actual"].to_numpy(),
    pred_B["actual"].to_numpy()
)

assert np.allclose(
    pred_A["actual"].to_numpy(),
    official_test_rows["response"].to_numpy(),
    atol=1e-5
)

# Build paired prediction dataset
paired = pd.DataFrame({
    "SIDM": official_test_rows["SIDM"].to_numpy(),
    "sex": official_test_rows["final_sex"].to_numpy(),
    "actual": pred_A["actual"].to_numpy(),
    "pred_A": pred_A["predicted"].to_numpy(),
    "pred_B": pred_B["predicted"].to_numpy()
})

def paired_bootstrap(df, n_boot=1000, seed=42):

    rng = np.random.default_rng(seed)

    groups = {
        sidm: group
        for sidm, group in df.groupby("SIDM")
    }

    ids = np.array(list(groups.keys()))

    differences = []

    for _ in range(n_boot):

        sampled = rng.choice(
            ids,
            size=len(ids),
            replace=True
        )

        boot = pd.concat(
            [groups[s] for s in sampled],
            ignore_index=True
        )

        rmse_A = np.sqrt(
            np.mean(
                (boot["actual"] - boot["pred_A"]) ** 2
            )
        )

        rmse_B = np.sqrt(
            np.mean(
                (boot["actual"] - boot["pred_B"]) ** 2
            )
        )

        differences.append(rmse_B - rmse_A)

    observed = (
        np.sqrt(np.mean((df["actual"] - df["pred_B"]) ** 2))
        - np.sqrt(np.mean((df["actual"] - df["pred_A"]) ** 2))
    )

    return {
        "Observed_difference": observed,
        "CI_lower": np.percentile(differences, 2.5),
        "CI_upper": np.percentile(differences, 97.5)
    }


comparison_rows = []

for group, subset in [
    ("Overall", paired),
    ("Female", paired[paired["sex"] == "Female"]),
    ("Male", paired[paired["sex"] == "Male"])
]:

    result = paired_bootstrap(subset)

    comparison_rows.append({
        "Group": group,
        **result,
        "CI_includes_zero": (
            result["CI_lower"] <= 0 <= result["CI_upper"]
        )
    })

official_B_vs_A = pd.DataFrame(comparison_rows)

print("=== OFFICIAL MODEL B MINUS MODEL A RMSE ===")
display(official_B_vs_A.round(4))

official_B_vs_A.to_csv(
    official_dir / "model_B_vs_A_bootstrap.csv",
    index=False
)

print("\nSaved: model_B_vs_A_bootstrap.csv")


=== OFFICIAL MODEL B MINUS MODEL A RMSE ===


,Group,Observed_difference,CI_lower,CI_upper,CI_includes_zero
0,Overall,-0.0103,-0.0391,0.0197,True
1,Female,0.0013,-0.0402,0.0433,True
2,Male,-0.0195,-0.0600,0.0192,True



Saved: model_B_vs_A_bootstrap.csv


In [32]:

# STEP 39: Prepare official tissue-blind experiment

import pandas as pd
from pathlib import Path

# Separate folder for tissue-blind experiments
tissue_dir = Path(
    "/content/week3_outputs/official_tissue_blind"
)
tissue_dir.mkdir(parents=True, exist_ok=True)

# Apply Malavika's official tissue-blind assignments
tissue_split_lookup = dict(
    zip(
        tissue_splits["cell_line_id"].astype(str),
        tissue_splits["split"].astype(str)
    )
)

# Make separate copies to preserve cell-blind results
tissue_eligible = eligible.copy()
tissue_response_pairs = response_pairs.copy()

tissue_eligible["split"] = (
    tissue_eligible["SIDM"]
    .astype(str)
    .map(tissue_split_lookup)
)

tissue_response_pairs["split"] = (
    tissue_response_pairs["SIDM"]
    .astype(str)
    .map(tissue_split_lookup)
)

# Verify all observations have assignments
assert tissue_eligible["split"].notna().all()
assert tissue_response_pairs["split"].notna().all()

# Create tissue-blind cell-line groups
tissue_train_cells = tissue_eligible[
    tissue_eligible["split"] == "train"
].copy()

tissue_val_cells = tissue_eligible[
    tissue_eligible["split"] == "val"
].copy()

tissue_test_cells = tissue_eligible[
    tissue_eligible["split"] == "test"
].copy()

# Verify no tissue category crosses partitions
tissue_overlap = tissue_splits.groupby(
    "tissue"
)["split"].nunique()

assert (tissue_overlap == 1).all(), (
    "Tissue leakage detected!"
)

print("=== OFFICIAL TISSUE-BLIND SPLITS ===")

print("\nCell lines:")
print("Training:", len(tissue_train_cells))
print("Validation:", len(tissue_val_cells))
print("Testing:", len(tissue_test_cells))

print("\nDrug-response observations:")
print(tissue_response_pairs["split"].value_counts())

print("\nSex distribution:")
print(
    tissue_eligible.groupby(
        ["split", "final_sex"]
    ).size()
)

print("\nHeld-out validation tissues:")
print(
    sorted(
        tissue_val_cells["Tissue_type"]
        .dropna()
        .unique()
        .tolist()
    )
)

print("\nHeld-out test tissues:")
print(
    sorted(
        tissue_test_cells["Tissue_type"]
        .dropna()
        .unique()
        .tolist()
    )
)

# Save the split assignments
tissue_response_pairs.to_csv(
    tissue_dir / "response_pairs_official_tissue_split.csv",
    index=False
)

print("\nTissue-blind split assignments saved!")


=== OFFICIAL TISSUE-BLIND SPLITS ===

Cell lines:
Training: 338
Validation: 74
Testing: 79

Drug-response observations:
split
train    74465
test     16256
val      15432
Name: count, dtype: int64

Sex distribution:
split  final_sex
test   Female        57
       Male          22
train  Female       125
       Male         213
val    Female        42
       Male          32
dtype: int64

Held-out validation tissues:
['Large Intestine', 'Ovary', 'Pancreas']

Held-out test tissues:
['Breast', 'Skin', 'Stomach']

Tissue-blind split assignments saved!


In [33]:

# STEP 40: Refit preprocessing for tissue-blind experiment

import numpy as np
from multiomics_preprocessing_pipeline import MultiOmicsPreprocessor

# Get tissue-blind training identifiers
tissue_train_ids = (
    tissue_train_cells["SIDM"]
    .astype(str)
    .tolist()
)

# All eligible cell lines
tissue_eligible_ids = (
    tissue_eligible["SIDM"]
    .astype(str)
    .tolist()
)

# Create a separate processor
tissue_processor = MultiOmicsPreprocessor(
    protein_min_coverage=0.70
)

# Fit ONLY on the 338 tissue-blind training cells
tissue_processor.fit(
    omics_matrices,
    tissue_train_ids
)

# Apply training-fitted transformations
tissue_processed_omics = tissue_processor.transform(
    omics_matrices,
    tissue_eligible_ids
)

print("=== TISSUE-BLIND PREPROCESSING COMPLETE ===")

for name, matrix in tissue_processed_omics.items():

    values = matrix.to_numpy(dtype=np.float32)

    print(
        f"{name}: {values.shape} | "
        f"All finite: {np.isfinite(values).all()}"
    )

print("\nFitted on training cells:", len(tissue_train_ids))
print("Total transformed cells:", len(tissue_eligible_ids))


=== TISSUE-BLIND PREPROCESSING COMPLETE ===
expression: (491, 17974) | All finite: True
mutation: (491, 1641) | All finite: True
cnv: (491, 23264) | All finite: True
proteomics: (491, 4162) | All finite: True

Fitted on training cells: 338
Total transformed cells: 491


In [34]:

# STEP 41: Build tissue-blind PyTorch inputs and loaders

import torch
import numpy as np
from torch.utils.data import DataLoader

# Preserve the tissue-blind cell-line ordering
tissue_cell_ids = (
    tissue_eligible["SIDM"].astype(str).tolist()
)

tissue_cell_to_idx = {
    sidm: i for i, sidm in enumerate(tissue_cell_ids)
}

# Create new molecular tensors
tissue_omics_tensors = {
    name: torch.tensor(
        tissue_processed_omics[name]
        .loc[tissue_cell_ids]
        .to_numpy(dtype=np.float32)
    )
    for name in tissue_processed_omics
}

# New feature dimensions
tissue_input_dims = {
    name: tensor.shape[1]
    for name, tensor in tissue_omics_tensors.items()
}

# Build tissue-blind observation table
tissue_model_pairs = tissue_response_pairs.copy()

tissue_model_pairs["cell_idx"] = (
    tissue_model_pairs["SIDM"]
    .astype(str)
    .map(tissue_cell_to_idx)
)

tissue_model_pairs["drug_idx"] = (
    tissue_model_pairs["Drug_ID"]
    .map(drug_to_idx)
)

tissue_model_pairs["sex_idx"] = (
    tissue_model_pairs["final_sex"]
    .map({"Female": 0, "Male": 1})
)

# Verify all mappings exist
assert tissue_model_pairs[
    ["cell_idx", "drug_idx", "sex_idx"]
].notna().all().all()

# Reuse your existing Dataset class
tissue_train_dataset = DrugResponseDataset(
    tissue_model_pairs[
        tissue_model_pairs["split"] == "train"
    ]
)

tissue_val_dataset = DrugResponseDataset(
    tissue_model_pairs[
        tissue_model_pairs["split"] == "val"
    ]
)

tissue_test_dataset = DrugResponseDataset(
    tissue_model_pairs[
        tissue_model_pairs["split"] == "test"
    ]
)

# Build new DataLoaders
tissue_train_loader = DataLoader(
    tissue_train_dataset,
    batch_size=256,
    shuffle=True
)

tissue_val_loader = DataLoader(
    tissue_val_dataset,
    batch_size=256,
    shuffle=False
)

tissue_test_loader = DataLoader(
    tissue_test_dataset,
    batch_size=256,
    shuffle=False
)

# Move new molecular tensors to GPU
tissue_omics_device = {
    name: tensor.to(device)
    for name, tensor in tissue_omics_tensors.items()
}

print("=== TISSUE-BLIND INPUTS READY ===")

print("\nFeature dimensions:")
print(tissue_input_dims)

print("\nDrug-response observations:")
print("Training:", len(tissue_train_dataset))
print("Validation:", len(tissue_val_dataset))
print("Testing:", len(tissue_test_dataset))

print("\nDevice:", device)

print(
    "All tensors updated:",
    all(
        tissue_omics_device[name].shape[1]
        == tissue_input_dims[name]
        for name in tissue_input_dims
    )
)


=== TISSUE-BLIND INPUTS READY ===

Feature dimensions:
{'expression': 17974, 'mutation': 1641, 'cnv': 23264, 'proteomics': 4162}

Drug-response observations:
Training: 74465
Validation: 15432
Testing: 16256

Device: cuda
All tensors updated: True


In [35]:

# STEP 42: Tissue-blind training function

import torch
import torch.nn as nn
import numpy as np
import copy

def train_fusion_model_tissue(
    model,
    train_loader,
    val_loader,
    omics_source,
    epochs=15,
    learning_rate=0.001,
    patience=4
):
    model = model.to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=learning_rate,
        weight_decay=1e-4
    )

    criterion = nn.MSELoss()

    best_val_rmse = float("inf")
    best_state = None
    patience_counter = 0
    history = []

    for epoch in range(epochs):

        # TRAINING
        model.train()
        train_squared_error = 0.0
        train_count = 0

        for cells, drugs, sexes, targets in train_loader:

            cells = cells.to(device)
            drugs = drugs.to(device)
            sexes = sexes.to(device)
            targets = targets.to(device)

            batch_omics = {
                name: values[cells]
                for name, values in omics_source.items()
                if name in model.modalities
            }

            optimizer.zero_grad()

            predictions = model(
                batch_omics,
                drugs,
                sexes if model.use_sex else None
            )

            loss = criterion(predictions, targets)
            loss.backward()
            optimizer.step()

            train_squared_error += (
                (predictions.detach() - targets)
                .square()
                .sum()
                .item()
            )

            train_count += len(targets)

        train_rmse = np.sqrt(
            train_squared_error / train_count
        )

        # VALIDATION
        model.eval()
        val_squared_error = 0.0
        val_count = 0

        with torch.no_grad():

            for cells, drugs, sexes, targets in val_loader:

                cells = cells.to(device)
                drugs = drugs.to(device)
                sexes = sexes.to(device)
                targets = targets.to(device)

                batch_omics = {
                    name: values[cells]
                    for name, values in omics_source.items()
                    if name in model.modalities
                }

                predictions = model(
                    batch_omics,
                    drugs,
                    sexes if model.use_sex else None
                )

                val_squared_error += (
                    (predictions - targets)
                    .square()
                    .sum()
                    .item()
                )

                val_count += len(targets)

        val_rmse = np.sqrt(
            val_squared_error / val_count
        )

        history.append({
            "epoch": epoch + 1,
            "train_rmse": train_rmse,
            "val_rmse": val_rmse
        })

        print(
            f"Epoch {epoch + 1:02d} | "
            f"Train RMSE: {train_rmse:.4f} | "
            f"Val RMSE: {val_rmse:.4f}"
        )

        # EARLY STOPPING
        if val_rmse < best_val_rmse - 1e-4:

            best_val_rmse = val_rmse

            best_state = copy.deepcopy(
                model.state_dict()
            )

            patience_counter = 0

        else:
            patience_counter += 1

            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    # Restore best checkpoint
    model.load_state_dict(best_state)

    return model, history, best_val_rmse


print("Tissue-blind training function ready!")

print(
    "Tissue-blind modalities:",
    list(tissue_omics_device.keys())
)

print("Device:", device)


Tissue-blind training function ready!
Tissue-blind modalities: ['expression', 'mutation', 'cnv', 'proteomics']
Device: cuda


In [36]:

# STEP 43: Train official tissue-blind Model A

import torch
import pandas as pd
from pathlib import Path

# Separate folder for tissue-blind results
tissue_dir = Path(
    "/content/week3_outputs/official_tissue_blind"
)
tissue_dir.mkdir(parents=True, exist_ok=True)

# Reproducibility
set_seed(42)

# Initialize a fresh model
tissue_model_A = OmicsAttentionFusion(
    input_dims=tissue_input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=False,
    use_sex=False
)

print("=== TISSUE-BLIND MODEL A ===")
print("Modalities:", tissue_model_A.modalities)
print("Sex-aware:", tissue_model_A.use_sex)
print("Training observations:", len(tissue_train_dataset))
print("Validation observations:", len(tissue_val_dataset))

# Train with tissue-blind molecular tensors
tissue_model_A, tissue_history_A, tissue_val_A = (
    train_fusion_model_tissue(
        model=tissue_model_A,
        train_loader=tissue_train_loader,
        val_loader=tissue_val_loader,
        omics_source=tissue_omics_device,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save model checkpoint
torch.save(
    tissue_model_A.state_dict(),
    tissue_dir / "model_A_baseline.pt"
)

# Save training history
pd.DataFrame(tissue_history_A).to_csv(
    tissue_dir / "model_A_training_history.csv",
    index=False
)

print("\n=== TISSUE-BLIND MODEL A COMPLETE ===")
print("Best validation RMSE:", round(tissue_val_A, 4))
print("Checkpoint saved successfully!")


=== TISSUE-BLIND MODEL A ===
Modalities: ['expression', 'mutation', 'cnv']
Sex-aware: False
Training observations: 74465
Validation observations: 15432
Epoch 01 | Train RMSE: 1.7942 | Val RMSE: 1.4064
Epoch 02 | Train RMSE: 1.2785 | Val RMSE: 1.4027
Epoch 03 | Train RMSE: 1.2254 | Val RMSE: 1.4056
Epoch 04 | Train RMSE: 1.1958 | Val RMSE: 1.4107
Epoch 05 | Train RMSE: 1.1771 | Val RMSE: 1.4104
Epoch 06 | Train RMSE: 1.1631 | Val RMSE: 1.4180
Early stopping triggered.

=== TISSUE-BLIND MODEL A COMPLETE ===
Best validation RMSE: 1.4027
Checkpoint saved successfully!


In [37]:

# STEP 44: Train tissue-blind Model B with proteomics

import torch
import pandas as pd

set_seed(42)

# Initialize a fresh model
tissue_model_B = OmicsAttentionFusion(
    input_dims=tissue_input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=True,
    use_sex=False
)

print("=== TISSUE-BLIND MODEL B ===")
print("Modalities:", tissue_model_B.modalities)
print("Sex-aware:", tissue_model_B.use_sex)

# Train using tissue-blind molecular features
tissue_model_B, tissue_history_B, tissue_val_B = (
    train_fusion_model_tissue(
        model=tissue_model_B,
        train_loader=tissue_train_loader,
        val_loader=tissue_val_loader,
        omics_source=tissue_omics_device,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save checkpoint
torch.save(
    tissue_model_B.state_dict(),
    tissue_dir / "model_B_proteomics.pt"
)

# Save training history
pd.DataFrame(tissue_history_B).to_csv(
    tissue_dir / "model_B_training_history.csv",
    index=False
)

print("\n=== TISSUE-BLIND VALIDATION COMPARISON ===")
print("Model A RMSE:", round(tissue_val_A, 4))
print("Model B RMSE:", round(tissue_val_B, 4))

difference = tissue_val_B - tissue_val_A

print("Difference (B - A):", round(difference, 4))

if difference < 0:
    print("Proteomics improved validation RMSE.")
elif difference > 0:
    print("Proteomics worsened validation RMSE.")
else:
    print("Validation RMSE was unchanged.")

print("\nTissue-blind Model B checkpoint saved!")


=== TISSUE-BLIND MODEL B ===
Modalities: ['expression', 'mutation', 'cnv', 'proteomics']
Sex-aware: False
Epoch 01 | Train RMSE: 1.7769 | Val RMSE: 1.4233
Epoch 02 | Train RMSE: 1.2702 | Val RMSE: 1.4041
Epoch 03 | Train RMSE: 1.2249 | Val RMSE: 1.4171
Epoch 04 | Train RMSE: 1.1972 | Val RMSE: 1.4055
Epoch 05 | Train RMSE: 1.1802 | Val RMSE: 1.4156
Epoch 06 | Train RMSE: 1.1661 | Val RMSE: 1.4071
Early stopping triggered.

=== TISSUE-BLIND VALIDATION COMPARISON ===
Model A RMSE: 1.4027
Model B RMSE: 1.4041
Difference (B - A): 0.0014
Proteomics worsened validation RMSE.

Tissue-blind Model B checkpoint saved!


In [38]:

# STEP 45: Train tissue-blind Model C with sex-aware input

import torch
import pandas as pd

set_seed(42)

# Initialize a fresh model
tissue_model_C = OmicsAttentionFusion(
    input_dims=tissue_input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=False,
    use_sex=True
)

print("=== TISSUE-BLIND MODEL C ===")
print("Modalities:", tissue_model_C.modalities)
print("Sex-aware:", tissue_model_C.use_sex)

# Train on tissue-blind molecular features
tissue_model_C, tissue_history_C, tissue_val_C = (
    train_fusion_model_tissue(
        model=tissue_model_C,
        train_loader=tissue_train_loader,
        val_loader=tissue_val_loader,
        omics_source=tissue_omics_device,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save checkpoint and history
torch.save(
    tissue_model_C.state_dict(),
    tissue_dir / "model_C_sex_aware.pt"
)

pd.DataFrame(tissue_history_C).to_csv(
    tissue_dir / "model_C_training_history.csv",
    index=False
)

print("\n=== TISSUE-BLIND VALIDATION COMPARISON ===")
print("Model A RMSE:", round(tissue_val_A, 4))
print("Model C RMSE:", round(tissue_val_C, 4))

difference = tissue_val_C - tissue_val_A

print("Difference (C - A):", round(difference, 4))

if difference < 0:
    print("Sex-aware input improved validation RMSE.")
elif difference > 0:
    print("Sex-aware input worsened validation RMSE.")
else:
    print("Validation RMSE was unchanged.")

print("\nTissue-blind Model C checkpoint saved!")


=== TISSUE-BLIND MODEL C ===
Modalities: ['expression', 'mutation', 'cnv']
Sex-aware: True
Epoch 01 | Train RMSE: 1.8056 | Val RMSE: 1.4246
Epoch 02 | Train RMSE: 1.2782 | Val RMSE: 1.4245
Epoch 03 | Train RMSE: 1.2276 | Val RMSE: 1.4111
Epoch 04 | Train RMSE: 1.1965 | Val RMSE: 1.4379
Epoch 05 | Train RMSE: 1.1841 | Val RMSE: 1.4435
Epoch 06 | Train RMSE: 1.1692 | Val RMSE: 1.4063
Epoch 07 | Train RMSE: 1.1566 | Val RMSE: 1.4337
Epoch 08 | Train RMSE: 1.1459 | Val RMSE: 1.4200
Epoch 09 | Train RMSE: 1.1414 | Val RMSE: 1.4251
Epoch 10 | Train RMSE: 1.1293 | Val RMSE: 1.4490
Early stopping triggered.

=== TISSUE-BLIND VALIDATION COMPARISON ===
Model A RMSE: 1.4027
Model C RMSE: 1.4063
Difference (C - A): 0.0035
Sex-aware input worsened validation RMSE.

Tissue-blind Model C checkpoint saved!


In [39]:

# STEP 46: Train tissue-blind Model D

import torch
import pandas as pd

set_seed(42)

# Model D: all four omics modalities + sex-aware input
tissue_model_D = OmicsAttentionFusion(
    input_dims=tissue_input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=True,
    use_sex=True
)

print("=== TISSUE-BLIND MODEL D ===")
print("Modalities:", tissue_model_D.modalities)
print("Sex-aware:", tissue_model_D.use_sex)

# Train with tissue-blind preprocessing
tissue_model_D, tissue_history_D, tissue_val_D = (
    train_fusion_model_tissue(
        model=tissue_model_D,
        train_loader=tissue_train_loader,
        val_loader=tissue_val_loader,
        omics_source=tissue_omics_device,
        epochs=15,
        learning_rate=0.001,
        patience=4
    )
)

# Save checkpoint and training history
torch.save(
    tissue_model_D.state_dict(),
    tissue_dir / "model_D_full_fusion.pt"
)

pd.DataFrame(tissue_history_D).to_csv(
    tissue_dir / "model_D_training_history.csv",
    index=False
)

print("\n=== TISSUE-BLIND FOUR-MODEL VALIDATION COMPARISON ===")

tissue_results = pd.DataFrame([
    {
        "Model": "A_baseline",
        "Proteomics": False,
        "Sex_aware": False,
        "Best_val_RMSE": tissue_val_A
    },
    {
        "Model": "B_proteomics",
        "Proteomics": True,
        "Sex_aware": False,
        "Best_val_RMSE": tissue_val_B
    },
    {
        "Model": "C_sex_aware",
        "Proteomics": False,
        "Sex_aware": True,
        "Best_val_RMSE": tissue_val_C
    },
    {
        "Model": "D_full",
        "Proteomics": True,
        "Sex_aware": True,
        "Best_val_RMSE": tissue_val_D
    }
])

tissue_results = tissue_results.sort_values(
    "Best_val_RMSE"
)

display(tissue_results.round(4))

tissue_results.to_csv(
    tissue_dir / "four_model_validation_comparison.csv",
    index=False
)

print("\nTissue-blind Model D checkpoint saved!")
print("Validation comparison saved!")


=== TISSUE-BLIND MODEL D ===
Modalities: ['expression', 'mutation', 'cnv', 'proteomics']
Sex-aware: True
Epoch 01 | Train RMSE: 1.8166 | Val RMSE: 1.4463
Epoch 02 | Train RMSE: 1.2790 | Val RMSE: 1.4126
Epoch 03 | Train RMSE: 1.2268 | Val RMSE: 1.4155
Epoch 04 | Train RMSE: 1.2015 | Val RMSE: 1.4209
Epoch 05 | Train RMSE: 1.1824 | Val RMSE: 1.4230
Epoch 06 | Train RMSE: 1.1680 | Val RMSE: 1.4755
Early stopping triggered.

=== TISSUE-BLIND FOUR-MODEL VALIDATION COMPARISON ===


,Model,Proteomics,Sex_aware,Best_val_RMSE
0,A_baseline,False,False,1.4027
1,B_proteomics,True,False,1.4041
2,C_sex_aware,False,True,1.4063
3,D_full,True,True,1.4126



Tissue-blind Model D checkpoint saved!
Validation comparison saved!


In [40]:

# STEP 47: Evaluate official tissue-blind models by sex

import torch
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

def evaluate_tissue_model(
    model, loader, model_name, omics_source
):
    model = model.to(device)
    model.eval()

    predictions = []
    targets_all = []
    sex_all = []

    with torch.no_grad():
        for cells, drugs, sexes, targets in loader:
            cells = cells.to(device)
            drugs = drugs.to(device)
            sexes = sexes.to(device)

            batch_omics = {
                name: values[cells]
                for name, values in omics_source.items()
                if name in model.modalities
            }

            pred = model(
                batch_omics,
                drugs,
                sexes if model.use_sex else None
            )

            predictions.extend(pred.cpu().numpy())
            targets_all.extend(targets.numpy())
            sex_all.extend(sexes.cpu().numpy())

    df = pd.DataFrame({
        "actual": targets_all,
        "predicted": predictions,
        "sex_idx": sex_all
    })

    df["sex"] = df["sex_idx"].map({
        0: "Female",
        1: "Male"
    })

    results = []

    for group, subset in [
        ("Overall", df),
        ("Female", df[df["sex"] == "Female"]),
        ("Male", df[df["sex"] == "Male"])
    ]:
        y = subset["actual"].to_numpy()
        p = subset["predicted"].to_numpy()

        rmse = np.sqrt(np.mean((y - p) ** 2))
        mae = np.mean(np.abs(y - p))

        if len(y) > 2 and np.std(y) > 0 and np.std(p) > 0:
            corr = pearsonr(y, p).statistic
        else:
            corr = np.nan

        results.append({
            "Model": model_name,
            "Group": group,
            "N_pairs": len(subset),
            "RMSE": float(rmse),
            "MAE": float(mae),
            "Pearson": float(corr)
        })

    return pd.DataFrame(results), df


# All four trained tissue-blind models
tissue_models = {
    "A_baseline": tissue_model_A,
    "B_proteomics": tissue_model_B,
    "C_sex_aware": tissue_model_C,
    "D_full": tissue_model_D
}

tissue_metrics_list = []
tissue_predictions = {}

for name, model in tissue_models.items():

    metrics_df, pred_df = evaluate_tissue_model(
        model,
        tissue_test_loader,
        name,
        tissue_omics_device
    )

    tissue_metrics_list.append(metrics_df)
    tissue_predictions[name] = pred_df

# Combine results
tissue_test_metrics = pd.concat(
    tissue_metrics_list,
    ignore_index=True
)

print("=== OFFICIAL TISSUE-BLIND TEST RESULTS ===")
display(tissue_test_metrics.round(4))

# Calculate sex-stratified RMSE gaps
tissue_gap_table = tissue_test_metrics.pivot(
    index="Model",
    columns="Group",
    values="RMSE"
)

tissue_gap_table["Female_minus_Male_RMSE"] = (
    tissue_gap_table["Female"] -
    tissue_gap_table["Male"]
)

print("\n=== TISSUE-BLIND SEX RMSE GAPS ===")
display(tissue_gap_table.round(4))

# Save results
tissue_test_metrics.to_csv(
    tissue_dir / "four_model_test_metrics_by_sex.csv",
    index=False
)

tissue_gap_table.to_csv(
    tissue_dir / "four_model_test_rmse_gaps.csv"
)

print("\nTissue-blind test results saved!")


=== OFFICIAL TISSUE-BLIND TEST RESULTS ===


,Model,Group,N_pairs,RMSE,MAE,Pearson
0,A_baseline,Overall,16256,1.4337,1.0674,0.8446
1,A_baseline,Female,12153,1.4380,1.0757,0.8441
2,A_baseline,Male,4103,1.4210,1.0426,0.8466
3,B_proteomics,Overall,16256,1.4225,1.0525,0.8480
4,B_proteomics,Female,12153,1.4211,1.0560,0.8489
5,B_proteomics,Male,4103,1.4268,1.0422,0.8453
6,C_sex_aware,Overall,16256,1.4375,1.0749,0.8441
7,C_sex_aware,Female,12153,1.4318,1.0761,0.8479
8,C_sex_aware,Male,4103,1.4545,1.0714,0.8360
9,D_full,Overall,16256,1.4369,1.0666,0.8441



=== TISSUE-BLIND SEX RMSE GAPS ===


Group,Female,Male,Overall,Female_minus_Male_RMSE
Model,,,,
A_baseline,1.4380,1.4210,1.4337,0.0171
B_proteomics,1.4211,1.4268,1.4225,-0.0057
C_sex_aware,1.4318,1.4545,1.4375,-0.0228
D_full,1.4346,1.4436,1.4369,-0.0090



Tissue-blind test results saved!


In [41]:

# STEP 48: Tissue-blind sex RMSE gap bootstrap

import numpy as np
import pandas as pd

# Recover test observations in DataLoader order
tissue_test_rows = tissue_model_pairs[
    tissue_model_pairs["split"] == "test"
].reset_index(drop=True)

def bootstrap_tissue_sex_gap(
    df, n_boot=1000, seed=42
):
    rng = np.random.default_rng(seed)

    # Group observations by cell line
    groups = {
        sidm: group
        for sidm, group in df.groupby("SIDM")
    }

    female_ids = df.loc[
        df["sex"] == "Female", "SIDM"
    ].unique()

    male_ids = df.loc[
        df["sex"] == "Male", "SIDM"
    ].unique()

    gaps = []

    for _ in range(n_boot):

        female_sample = rng.choice(
            female_ids,
            size=len(female_ids),
            replace=True
        )

        male_sample = rng.choice(
            male_ids,
            size=len(male_ids),
            replace=True
        )

        female_df = pd.concat(
            [groups[s] for s in female_sample],
            ignore_index=True
        )

        male_df = pd.concat(
            [groups[s] for s in male_sample],
            ignore_index=True
        )

        female_rmse = np.sqrt(
            np.mean(
                (female_df["actual"] -
                 female_df["predicted"]) ** 2
            )
        )

        male_rmse = np.sqrt(
            np.mean(
                (male_df["actual"] -
                 male_df["predicted"]) ** 2
            )
        )

        gaps.append(
            female_rmse - male_rmse
        )

    return np.percentile(
        gaps,
        [2.5, 50, 97.5]
    )


bootstrap_results = []

for name, pred_df in tissue_predictions.items():

    # Confirm predictions align with test observations
    assert len(pred_df) == len(tissue_test_rows)

    assert np.allclose(
        pred_df["actual"].to_numpy(),
        tissue_test_rows["response"].to_numpy(),
        atol=1e-5
    ), f"Ordering mismatch for {name}"

    combined = pred_df.copy()

    combined["SIDM"] = (
        tissue_test_rows["SIDM"].to_numpy()
    )

    lower, median, upper = bootstrap_tissue_sex_gap(
        combined,
        n_boot=1000,
        seed=42
    )

    bootstrap_results.append({
        "Model": name,
        "Median_gap": median,
        "CI_lower": lower,
        "CI_upper": upper,
        "CI_includes_zero": lower <= 0 <= upper
    })

tissue_bootstrap = pd.DataFrame(
    bootstrap_results
)

print("=== TISSUE-BLIND SEX GAP BOOTSTRAP ===")
display(tissue_bootstrap.round(4))

tissue_bootstrap.to_csv(
    tissue_dir / "sex_rmse_gap_bootstrap.csv",
    index=False
)

print("\nTissue-blind bootstrap results saved!")


=== TISSUE-BLIND SEX GAP BOOTSTRAP ===


,Model,Median_gap,CI_lower,CI_upper,CI_includes_zero
0,A_baseline,0.0191,-0.1218,0.1512,True
1,B_proteomics,-0.0038,-0.1404,0.1209,True
2,C_sex_aware,-0.0192,-0.1551,0.1164,True
3,D_full,-0.0064,-0.1430,0.1148,True



Tissue-blind bootstrap results saved!


In [42]:

# STEP 49: Tissue-blind paired bootstrap, Model B vs A

import numpy as np
import pandas as pd

pred_A = tissue_predictions["A_baseline"]
pred_B = tissue_predictions["B_proteomics"]

assert len(pred_A) == len(pred_B) == len(tissue_test_rows)

assert np.allclose(
    pred_A["actual"].to_numpy(),
    pred_B["actual"].to_numpy()
)

assert np.allclose(
    pred_A["actual"].to_numpy(),
    tissue_test_rows["response"].to_numpy(),
    atol=1e-5
)

paired_tissue = pd.DataFrame({
    "SIDM": tissue_test_rows["SIDM"].to_numpy(),
    "sex": tissue_test_rows["final_sex"].to_numpy(),
    "actual": pred_A["actual"].to_numpy(),
    "pred_A": pred_A["predicted"].to_numpy(),
    "pred_B": pred_B["predicted"].to_numpy()
})

def paired_tissue_bootstrap(df, n_boot=1000, seed=42):

    rng = np.random.default_rng(seed)

    groups = {
        sidm: group
        for sidm, group in df.groupby("SIDM")
    }

    ids = np.array(list(groups.keys()))
    differences = []

    for _ in range(n_boot):

        sampled = rng.choice(
            ids,
            size=len(ids),
            replace=True
        )

        boot = pd.concat(
            [groups[s] for s in sampled],
            ignore_index=True
        )

        rmse_A = np.sqrt(
            np.mean(
                (boot["actual"] - boot["pred_A"]) ** 2
            )
        )

        rmse_B = np.sqrt(
            np.mean(
                (boot["actual"] - boot["pred_B"]) ** 2
            )
        )

        differences.append(rmse_B - rmse_A)

    observed = (
        np.sqrt(np.mean((df["actual"] - df["pred_B"]) ** 2))
        - np.sqrt(np.mean((df["actual"] - df["pred_A"]) ** 2))
    )

    return {
        "Observed_difference": observed,
        "CI_lower": np.percentile(differences, 2.5),
        "CI_upper": np.percentile(differences, 97.5)
    }


comparison_rows = []

for group, subset in [
    ("Overall", paired_tissue),
    ("Female", paired_tissue[paired_tissue["sex"] == "Female"]),
    ("Male", paired_tissue[paired_tissue["sex"] == "Male"])
]:

    result = paired_tissue_bootstrap(subset)

    comparison_rows.append({
        "Group": group,
        **result,
        "CI_includes_zero": (
            result["CI_lower"] <= 0 <= result["CI_upper"]
        )
    })

tissue_B_vs_A = pd.DataFrame(comparison_rows)

print("=== TISSUE-BLIND MODEL B MINUS MODEL A RMSE ===")
display(tissue_B_vs_A.round(4))

tissue_B_vs_A.to_csv(
    tissue_dir / "model_B_vs_A_bootstrap.csv",
    index=False
)

print("\nSaved: model_B_vs_A_bootstrap.csv")


=== TISSUE-BLIND MODEL B MINUS MODEL A RMSE ===


,Group,Observed_difference,CI_lower,CI_upper,CI_includes_zero
0,Overall,-0.0112,-0.0291,0.0045,True
1,Female,-0.0170,-0.0383,0.0023,True
2,Male,0.0058,-0.0181,0.0387,True



Saved: model_B_vs_A_bootstrap.csv


In [43]:

# STEP 50: Package official Week 3 results

from pathlib import Path
import zipfile

output_dir = Path("/content/week3_outputs")

zip_path = Path(
    "/content/week3_official_results.zip"
)

# Only package official experiments
official_folders = [
    output_dir / "official_cell_blind",
    output_dir / "official_tissue_blind"
]

files_to_save = []

for folder in official_folders:
    files_to_save.extend(
        p for p in folder.rglob("*")
        if p.is_file()
    )

files_to_save = sorted(files_to_save)

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:
    for path in files_to_save:
        z.write(
            path,
            arcname=str(path.relative_to(output_dir))
        )

print("=== OFFICIAL WEEK 3 PACKAGE ===")
print("Files included:", len(files_to_save))

for path in files_to_save:
    print("-", path.relative_to(output_dir))

print(
    "\nZIP size (MB):",
    round(
        zip_path.stat().st_size / (1024 ** 2),
        2
    )
)

print("\nPackage created:", zip_path)


=== OFFICIAL WEEK 3 PACKAGE ===
Files included: 27
- official_cell_blind/four_model_test_metrics_by_sex.csv
- official_cell_blind/four_model_test_rmse_gaps.csv
- official_cell_blind/four_model_validation_comparison.csv
- official_cell_blind/model_A_baseline.pt
- official_cell_blind/model_A_training_history.csv
- official_cell_blind/model_B_proteomics.pt
- official_cell_blind/model_B_training_history.csv
- official_cell_blind/model_B_vs_A_bootstrap.csv
- official_cell_blind/model_C_sex_aware.pt
- official_cell_blind/model_C_training_history.csv
- official_cell_blind/model_D_full_fusion.pt
- official_cell_blind/model_D_training_history.csv
- official_cell_blind/sex_rmse_gap_bootstrap.csv
- official_tissue_blind/four_model_test_metrics_by_sex.csv
- official_tissue_blind/four_model_test_rmse_gaps.csv
- official_tissue_blind/four_model_validation_comparison.csv
- official_tissue_blind/model_A_baseline.pt
- official_tissue_blind/model_A_training_history.csv
- official_tissue_blind/model_B_pr

In [44]:

from google.colab import files

files.download(
    "/content/week3_official_results.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [45]:

# STEP 52: Check readiness for multiple-seed experiments

required_variables = [
    "official_model_A",
    "official_model_B",
    "official_model_C",
    "official_model_D",
    "tissue_model_A",
    "tissue_model_B",
    "tissue_model_C",
    "tissue_model_D",
    "train_loader",
    "val_loader",
    "tissue_train_loader",
    "tissue_val_loader",
    "omics_device",
    "tissue_omics_device",
    "input_dims",
    "tissue_input_dims",
    "drug_ids",
    "official_dir",
    "tissue_dir"
]

missing = [
    name for name in required_variables
    if name not in globals()
]

print("=== MULTIPLE-SEED READINESS CHECK ===")

if not missing:
    print("All required variables are available!")
    print("Ready for additional training seeds.")
else:
    print("Missing variables:")
    for name in missing:
        print("-", name)


=== MULTIPLE-SEED READINESS CHECK ===
All required variables are available!
Ready for additional training seeds.


In [46]:

# STEP 53: Multiple-seed training for official experiments

import torch
import pandas as pd
from pathlib import Path

# Additional seeds; seed 42 was already trained
new_seeds = [123, 456]

model_configs = {
    "A_baseline": {
        "use_proteomics": False,
        "use_sex": False
    },
    "B_proteomics": {
        "use_proteomics": True,
        "use_sex": False
    },
    "C_sex_aware": {
        "use_proteomics": False,
        "use_sex": True
    },
    "D_full": {
        "use_proteomics": True,
        "use_sex": True
    }
}

experiments = {
    "cell_blind": {
        "input_dims": input_dims,
        "train_loader": train_loader,
        "val_loader": val_loader,
        "omics_source": omics_device,
        "output_dir": official_dir
    },
    "tissue_blind": {
        "input_dims": tissue_input_dims,
        "train_loader": tissue_train_loader,
        "val_loader": tissue_val_loader,
        "omics_source": tissue_omics_device,
        "output_dir": tissue_dir
    }
}

multi_seed_results = []

for split_name, experiment in experiments.items():

    print(f"\n{'=' * 55}")
    print(f"EXPERIMENT: {split_name.upper()}")
    print(f"{'=' * 55}")

    for seed in new_seeds:

        for model_name, config in model_configs.items():

            print(
                f"\nTraining {model_name} | "
                f"Seed {seed} | {split_name}"
            )

            set_seed(seed)

            model = OmicsAttentionFusion(
                input_dims=experiment["input_dims"],
                n_drugs=len(drug_ids),
                hidden_dim=64,
                use_proteomics=config["use_proteomics"],
                use_sex=config["use_sex"]
            )

            trained_model, history, best_val = (
                train_fusion_model_tissue(
                    model=model,
                    train_loader=experiment["train_loader"],
                    val_loader=experiment["val_loader"],
                    omics_source=experiment["omics_source"],
                    epochs=15,
                    learning_rate=0.001,
                    patience=4
                )
            )

            # Save each seed separately
            checkpoint_path = (
                experiment["output_dir"]
                / f"{model_name}_seed{seed}.pt"
            )

            torch.save(
                trained_model.state_dict(),
                checkpoint_path
            )

            pd.DataFrame(history).to_csv(
                experiment["output_dir"]
                / f"{model_name}_seed{seed}_history.csv",
                index=False
            )

            multi_seed_results.append({
                "Split": split_name,
                "Seed": seed,
                "Model": model_name,
                "Best_val_RMSE": best_val
            })

            print(
                f"Finished: {model_name}, "
                f"seed={seed}, "
                f"validation RMSE={best_val:.4f}"
            )

            # Free GPU memory before next model
            del model, trained_model
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

# Save summary
multi_seed_df = pd.DataFrame(multi_seed_results)

multi_seed_df.to_csv(
    "/content/week3_outputs/additional_seed_results.csv",
    index=False
)

print("\n=== ADDITIONAL SEED RESULTS ===")
display(multi_seed_df.round(4))

print("\nAll additional training runs completed!")



EXPERIMENT: CELL_BLIND

Training A_baseline | Seed 123 | cell_blind
Epoch 01 | Train RMSE: 1.7801 | Val RMSE: 1.3679
Epoch 02 | Train RMSE: 1.3002 | Val RMSE: 1.3628
Epoch 03 | Train RMSE: 1.2578 | Val RMSE: 1.3318
Epoch 04 | Train RMSE: 1.2300 | Val RMSE: 1.3435
Epoch 05 | Train RMSE: 1.2097 | Val RMSE: 1.3118
Epoch 06 | Train RMSE: 1.1949 | Val RMSE: 1.3179
Epoch 07 | Train RMSE: 1.1813 | Val RMSE: 1.3324
Epoch 08 | Train RMSE: 1.1729 | Val RMSE: 1.3226
Epoch 09 | Train RMSE: 1.1606 | Val RMSE: 1.3225
Early stopping triggered.
Finished: A_baseline, seed=123, validation RMSE=1.3118

Training B_proteomics | Seed 123 | cell_blind
Epoch 01 | Train RMSE: 1.7927 | Val RMSE: 1.3647
Epoch 02 | Train RMSE: 1.3009 | Val RMSE: 1.3279
Epoch 03 | Train RMSE: 1.2547 | Val RMSE: 1.3057
Epoch 04 | Train RMSE: 1.2273 | Val RMSE: 1.3062
Epoch 05 | Train RMSE: 1.2062 | Val RMSE: 1.3190
Epoch 06 | Train RMSE: 1.1929 | Val RMSE: 1.3299
Epoch 07 | Train RMSE: 1.1795 | Val RMSE: 1.3027
Epoch 08 | Train RM

,Split,Seed,Model,Best_val_RMSE
0,cell_blind,123,A_baseline,1.3118
1,cell_blind,123,B_proteomics,1.2959
2,cell_blind,123,C_sex_aware,1.3157
3,cell_blind,123,D_full,1.2915
4,cell_blind,456,A_baseline,1.3017
5,cell_blind,456,B_proteomics,1.2995
6,cell_blind,456,C_sex_aware,1.3160
7,cell_blind,456,D_full,1.2982
8,tissue_blind,123,A_baseline,1.4108
9,tissue_blind,123,B_proteomics,1.4077



All additional training runs completed!


In [47]:

# STEP 54: Evaluate all 24 official model runs

import torch
import pandas as pd
import numpy as np
from pathlib import Path

seeds = [42, 123, 456]

configs = {
    "A_baseline": (False, False),
    "B_proteomics": (True, False),
    "C_sex_aware": (False, True),
    "D_full": (True, True)
}

experiments = {
    "cell_blind": {
        "dims": input_dims,
        "loader": test_loader,
        "omics": omics_device,
        "directory": official_dir,
        "rows": model_pairs[
            model_pairs["split"] == "test"
        ].reset_index(drop=True),
        "seed42_models": {
            "A_baseline": official_model_A,
            "B_proteomics": official_model_B,
            "C_sex_aware": official_model_C,
            "D_full": official_model_D
        }
    },
    "tissue_blind": {
        "dims": tissue_input_dims,
        "loader": tissue_test_loader,
        "omics": tissue_omics_device,
        "directory": tissue_dir,
        "rows": tissue_model_pairs[
            tissue_model_pairs["split"] == "test"
        ].reset_index(drop=True),
        "seed42_models": {
            "A_baseline": tissue_model_A,
            "B_proteomics": tissue_model_B,
            "C_sex_aware": tissue_model_C,
            "D_full": tissue_model_D
        }
    }
}

all_seed_test_metrics = []

for split_name, exp in experiments.items():

    for seed in seeds:

        for model_name, (use_prot, use_sex) in configs.items():

            if seed == 42:
                model = exp["seed42_models"][model_name]
            else:
                model = OmicsAttentionFusion(
                    input_dims=exp["dims"],
                    n_drugs=len(drug_ids),
                    hidden_dim=64,
                    use_proteomics=use_prot,
                    use_sex=use_sex
                )

                checkpoint = (
                    exp["directory"]
                    / f"{model_name}_seed{seed}.pt"
                )

                model.load_state_dict(
                    torch.load(
                        checkpoint,
                        map_location=device,
                        weights_only=True
                    )
                )

            # Use explicit molecular tensors for each split
            metrics, predictions = evaluate_tissue_model(
                model,
                exp["loader"],
                model_name,
                exp["omics"]
            )

            assert len(predictions) == len(exp["rows"])
            assert np.allclose(
                predictions["actual"].to_numpy(),
                exp["rows"]["response"].to_numpy(),
                atol=1e-5
            )

            metrics["Split"] = split_name
            metrics["Seed"] = seed

            all_seed_test_metrics.append(metrics)

            # Save predictions for later paired analyses
            predictions["SIDM"] = exp["rows"]["SIDM"].values
            predictions["Drug_ID"] = exp["rows"]["Drug_ID"].values

            predictions.to_csv(
                exp["directory"]
                / f"{model_name}_seed{seed}_predictions.csv",
                index=False
            )

            print(
                f"Evaluated {split_name} | "
                f"{model_name} | seed {seed}"
            )

            if seed != 42:
                del model

# Combine all metrics
seed_test_df = pd.concat(
    all_seed_test_metrics,
    ignore_index=True
)

# Save detailed metrics
seed_test_df.to_csv(
    "/content/week3_outputs/all_seed_test_metrics.csv",
    index=False
)

# Calculate mean and standard deviation across seeds
summary = (
    seed_test_df
    .groupby(["Split", "Model", "Group"])
    .agg(
        Mean_RMSE=("RMSE", "mean"),
        SD_RMSE=("RMSE", "std"),
        Mean_MAE=("MAE", "mean"),
        Mean_Pearson=("Pearson", "mean")
    )
    .reset_index()
)

summary.to_csv(
    "/content/week3_outputs/"
    "three_seed_test_summary.csv",
    index=False
)

print("\n=== THREE-SEED TEST SUMMARY ===")
display(summary.round(4))

print("\nAll 24 evaluations completed!")
print("Detailed metrics and predictions saved.")


Evaluated cell_blind | A_baseline | seed 42
Evaluated cell_blind | B_proteomics | seed 42
Evaluated cell_blind | C_sex_aware | seed 42
Evaluated cell_blind | D_full | seed 42
Evaluated cell_blind | A_baseline | seed 123
Evaluated cell_blind | B_proteomics | seed 123
Evaluated cell_blind | C_sex_aware | seed 123
Evaluated cell_blind | D_full | seed 123
Evaluated cell_blind | A_baseline | seed 456
Evaluated cell_blind | B_proteomics | seed 456
Evaluated cell_blind | C_sex_aware | seed 456
Evaluated cell_blind | D_full | seed 456
Evaluated tissue_blind | A_baseline | seed 42
Evaluated tissue_blind | B_proteomics | seed 42
Evaluated tissue_blind | C_sex_aware | seed 42
Evaluated tissue_blind | D_full | seed 42
Evaluated tissue_blind | A_baseline | seed 123
Evaluated tissue_blind | B_proteomics | seed 123
Evaluated tissue_blind | C_sex_aware | seed 123
Evaluated tissue_blind | D_full | seed 123
Evaluated tissue_blind | A_baseline | seed 456
Evaluated tissue_blind | B_proteomics | seed 456
E

,Split,Model,Group,Mean_RMSE,SD_RMSE,Mean_MAE,Mean_Pearson
0,cell_blind,A_baseline,Female,1.3772,0.0109,1.0245,0.8639
1,cell_blind,A_baseline,Male,1.3297,0.0127,0.9982,0.8689
2,cell_blind,A_baseline,Overall,1.3507,0.0088,1.0097,0.8667
3,cell_blind,B_proteomics,Female,1.3560,0.0117,1.0102,0.8685
4,cell_blind,B_proteomics,Male,1.3202,0.0011,0.9910,0.8706
5,cell_blind,B_proteomics,Overall,1.3360,0.0049,0.9994,0.8696
6,cell_blind,C_sex_aware,Female,1.3732,0.0090,1.0220,0.8646
7,cell_blind,C_sex_aware,Male,1.3229,0.0047,0.9941,0.8694
8,cell_blind,C_sex_aware,Overall,1.3452,0.0037,1.0063,0.8671
9,cell_blind,D_full,Female,1.3618,0.0154,1.0094,0.8665



All 24 evaluations completed!
Detailed metrics and predictions saved.


In [48]:

# STEP 55: Paired seed-by-seed proteomics comparison

import pandas as pd
from pathlib import Path

# Use metrics already calculated in Step 54
ab_results = seed_test_df[
    seed_test_df["Model"].isin([
        "A_baseline",
        "B_proteomics"
    ])
].copy()

# Match A and B on split, seed, and sex group
ab_pivot = ab_results.pivot(
    index=["Split", "Seed", "Group"],
    columns="Model",
    values="RMSE"
).reset_index()

# Calculate paired RMSE difference
ab_pivot["B_minus_A"] = (
    ab_pivot["B_proteomics"]
    - ab_pivot["A_baseline"]
)

ab_pivot["Proteomics_improved"] = (
    ab_pivot["B_minus_A"] < 0
)

print("=== PROTEOMICS EFFECT BY SEED ===")
display(ab_pivot.round(4))

# Summarize consistency across the three seeds
consistency = (
    ab_pivot
    .groupby(["Split", "Group"])
    .agg(
        Mean_RMSE_Difference=("B_minus_A", "mean"),
        SD_Difference=("B_minus_A", "std"),
        Seeds_Improved=("Proteomics_improved", "sum"),
        Total_Seeds=("Seed", "count")
    )
    .reset_index()
)

print("\n=== THREE-SEED CONSISTENCY SUMMARY ===")
display(consistency.round(4))

# Save for final research documentation
output_dir = Path("/content/week3_outputs")

ab_pivot.to_csv(
    output_dir / "proteomics_effect_by_seed.csv",
    index=False
)

consistency.to_csv(
    output_dir / "proteomics_seed_consistency.csv",
    index=False
)

print("\nSeed comparison files saved!")


=== PROTEOMICS EFFECT BY SEED ===


Model,Split,Seed,Group,A_baseline,B_proteomics,B_minus_A,Proteomics_improved
0,cell_blind,42,Female,1.3681,1.3694,0.0013,False
1,cell_blind,42,Male,1.3391,1.3196,-0.0195,True
2,cell_blind,42,Overall,1.3519,1.3417,-0.0103,True
3,cell_blind,123,Female,1.3894,1.3491,-0.0403,True
4,cell_blind,123,Male,1.3347,1.3215,-0.0132,True
5,cell_blind,123,Overall,1.3589,1.3337,-0.0252,True
6,cell_blind,456,Female,1.3742,1.3493,-0.0249,True
7,cell_blind,456,Male,1.3153,1.3195,0.0043,False
8,cell_blind,456,Overall,1.3414,1.3326,-0.0088,True
9,tissue_blind,42,Female,1.4380,1.4211,-0.0170,True



=== THREE-SEED CONSISTENCY SUMMARY ===


,Split,Group,Mean_RMSE_Difference,SD_Difference,Seeds_Improved,Total_Seeds
0,cell_blind,Female,-0.0213,0.0210,2,3
1,cell_blind,Male,-0.0095,0.0123,2,3
2,cell_blind,Overall,-0.0148,0.0091,3,3
3,tissue_blind,Female,-0.0150,0.0054,3,3
4,tissue_blind,Male,-0.0120,0.0215,2,3
5,tissue_blind,Overall,-0.0143,0.0080,3,3



Seed comparison files saved!


In [49]:

# STEP 56: Package complete multi-seed Week 3 results

from pathlib import Path
import zipfile

output_dir = Path("/content/week3_outputs")

zip_path = Path(
    "/content/week3_complete_multiseed_results.zip"
)

# Official experiment folders
folders = [
    output_dir / "official_cell_blind",
    output_dir / "official_tissue_blind"
]

# Root-level multi-seed summaries
summary_files = [
    "additional_seed_results.csv",
    "all_seed_test_metrics.csv",
    "three_seed_test_summary.csv",
    "proteomics_effect_by_seed.csv",
    "proteomics_seed_consistency.csv"
]

files_to_save = []

for folder in folders:
    files_to_save.extend(
        p for p in folder.rglob("*")
        if p.is_file()
    )

for filename in summary_files:
    path = output_dir / filename

    if not path.exists():
        raise FileNotFoundError(
            f"Missing required summary: {filename}"
        )

    files_to_save.append(path)

files_to_save = sorted(set(files_to_save))

with zipfile.ZipFile(
    zip_path,
    "w",
    zipfile.ZIP_DEFLATED
) as z:

    for path in files_to_save:
        z.write(
            path,
            arcname=str(path.relative_to(output_dir))
        )

# Verify ZIP integrity
with zipfile.ZipFile(zip_path) as z:
    bad_file = z.testzip()
    assert bad_file is None, (
        f"ZIP integrity error: {bad_file}"
    )

print("=== COMPLETE WEEK 3 RESULTS PACKAGE ===")
print("Total files:", len(files_to_save))

for folder in folders:
    checkpoints = list(folder.glob("*.pt"))
    print(
        folder.name,
        "- checkpoints:",
        len(checkpoints)
    )

print(
    "ZIP size (MB):",
    round(zip_path.stat().st_size / 1024**2, 2)
)

print("ZIP integrity: PASSED")
print("Saved:", zip_path)


=== COMPLETE WEEK 3 RESULTS PACKAGE ===
Total files: 88
official_cell_blind - checkpoints: 12
official_tissue_blind - checkpoints: 12
ZIP size (MB): 506.76
ZIP integrity: PASSED
Saved: /content/week3_complete_multiseed_results.zip


In [50]:

from google.colab import files

files.download(
    "/content/week3_complete_multiseed_results.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [2]:

from pathlib import Path
import pandas as pd

set_seed(42)

# Model A: expression + mutation + CNV
model_A = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=False,
    use_sex=False
)

print("Training Model A...")
print("Modalities:", model_A.modalities)

model_A, history_A, best_val_A = train_fusion_model(
    model=model_A,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=15,
    learning_rate=0.001,
    patience=4
)

# Save checkpoint and training history
output_dir = Path("/content/week3_outputs")
output_dir.mkdir(exist_ok=True)

torch.save(
    model_A.state_dict(),
    output_dir / "model_A_baseline.pt"
)

pd.DataFrame(history_A).to_csv(
    output_dir / "model_A_training_history.csv",
    index=False
)

print("\nModel A training finished!")
print("Best validation RMSE:", round(best_val_A, 4))
print("Checkpoint saved: model_A_baseline.pt")


NameError: name 'set_seed' is not defined

In [17]:

from pathlib import Path
import pandas as pd
import torch

set_seed(42)

# Model B: expression + mutation + CNV + proteomics
model_B = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=True,
    use_sex=False
)

print("Training Model B...")
print("Modalities:", model_B.modalities)

model_B, history_B, best_val_B = train_fusion_model(
    model=model_B,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=15,
    learning_rate=0.001,
    patience=4
)

# Save model and training history
output_dir = Path("/content/week3_outputs")
output_dir.mkdir(exist_ok=True)

torch.save(
    model_B.state_dict(),
    output_dir / "model_B_proteomics.pt"
)

pd.DataFrame(history_B).to_csv(
    output_dir / "model_B_training_history.csv",
    index=False
)

# Compare validation performance
print("\n=== VALIDATION COMPARISON ===")
print("Model A RMSE:", round(best_val_A, 4))
print("Model B RMSE:", round(best_val_B, 4))

difference = best_val_B - best_val_A

print("Difference (B - A):", round(difference, 4))

if difference < 0:
    print("Adding proteomics improved validation RMSE.")
elif difference > 0:
    print("Adding proteomics worsened validation RMSE.")
else:
    print("Both models have equal validation RMSE.")

print("\nModel B checkpoint saved!")


Training Model B...
Modalities: ['expression', 'mutation', 'cnv', 'proteomics']
Epoch 01 | Train RMSE: 1.7625 | Val RMSE: 1.3861
Epoch 02 | Train RMSE: 1.3012 | Val RMSE: 1.3556
Epoch 03 | Train RMSE: 1.2532 | Val RMSE: 1.3667
Epoch 04 | Train RMSE: 1.2251 | Val RMSE: 1.3810
Epoch 05 | Train RMSE: 1.2068 | Val RMSE: 1.3398
Epoch 06 | Train RMSE: 1.1891 | Val RMSE: 1.3429
Epoch 07 | Train RMSE: 1.1818 | Val RMSE: 1.3518
Epoch 08 | Train RMSE: 1.1709 | Val RMSE: 1.3550
Epoch 09 | Train RMSE: 1.1627 | Val RMSE: 1.3272
Epoch 10 | Train RMSE: 1.1486 | Val RMSE: 1.3376
Epoch 11 | Train RMSE: 1.1362 | Val RMSE: 1.3363
Epoch 12 | Train RMSE: 1.1287 | Val RMSE: 1.3422
Epoch 13 | Train RMSE: 1.1165 | Val RMSE: 1.3409
Early stopping triggered.

=== VALIDATION COMPARISON ===
Model A RMSE: 1.2953
Model B RMSE: 1.3272
Difference (B - A): 0.0319
Adding proteomics worsened validation RMSE.

Model B checkpoint saved!


In [18]:

from pathlib import Path
import pandas as pd
import torch

set_seed(42)

# Model C: no proteomics, with sex-aware input
model_C = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=False,
    use_sex=True
)

print("Training Model C...")
print("Modalities:", model_C.modalities)
print("Sex-aware input:", model_C.use_sex)

model_C, history_C, best_val_C = train_fusion_model(
    model=model_C,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=15,
    learning_rate=0.001,
    patience=4
)

# Save checkpoint and training history
output_dir = Path("/content/week3_outputs")
output_dir.mkdir(exist_ok=True)

torch.save(
    model_C.state_dict(),
    output_dir / "model_C_sex_aware.pt"
)

pd.DataFrame(history_C).to_csv(
    output_dir / "model_C_training_history.csv",
    index=False
)

print("\n=== VALIDATION COMPARISON ===")
print("Model A (baseline):", round(best_val_A, 4))
print("Model C (sex-aware):", round(best_val_C, 4))

difference = best_val_C - best_val_A
print("Difference (C - A):", round(difference, 4))

if difference < 0:
    print("Sex-aware input improved validation RMSE.")
elif difference > 0:
    print("Sex-aware input worsened validation RMSE.")
else:
    print("Validation RMSE was unchanged.")

print("\nModel C checkpoint saved!")


Training Model C...
Modalities: ['expression', 'mutation', 'cnv']
Sex-aware input: True
Epoch 01 | Train RMSE: 1.8472 | Val RMSE: 1.4021
Epoch 02 | Train RMSE: 1.3109 | Val RMSE: 1.3389
Epoch 03 | Train RMSE: 1.2637 | Val RMSE: 1.3410
Epoch 04 | Train RMSE: 1.2281 | Val RMSE: 1.3259
Epoch 05 | Train RMSE: 1.2095 | Val RMSE: 1.3209
Epoch 06 | Train RMSE: 1.1943 | Val RMSE: 1.3365
Epoch 07 | Train RMSE: 1.1835 | Val RMSE: 1.3268
Epoch 08 | Train RMSE: 1.1704 | Val RMSE: 1.3199
Epoch 09 | Train RMSE: 1.1565 | Val RMSE: 1.3181
Epoch 10 | Train RMSE: 1.1456 | Val RMSE: 1.3281
Epoch 11 | Train RMSE: 1.1353 | Val RMSE: 1.3322
Epoch 12 | Train RMSE: 1.1257 | Val RMSE: 1.3231
Epoch 13 | Train RMSE: 1.1147 | Val RMSE: 1.3310
Early stopping triggered.

=== VALIDATION COMPARISON ===
Model A (baseline): 1.2953
Model C (sex-aware): 1.3181
Difference (C - A): 0.0228
Sex-aware input worsened validation RMSE.

Model C checkpoint saved!


In [19]:

from pathlib import Path
import pandas as pd
import torch

set_seed(42)

# Model D: all four omics layers + sex-aware input
model_D = OmicsAttentionFusion(
    input_dims=input_dims,
    n_drugs=len(drug_ids),
    hidden_dim=64,
    use_proteomics=True,
    use_sex=True
)

print("Training Model D...")
print("Modalities:", model_D.modalities)
print("Sex-aware input:", model_D.use_sex)

model_D, history_D, best_val_D = train_fusion_model(
    model=model_D,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=15,
    learning_rate=0.001,
    patience=4
)

# Save checkpoint and training history
output_dir = Path("/content/week3_outputs")
output_dir.mkdir(exist_ok=True)

torch.save(
    model_D.state_dict(),
    output_dir / "model_D_full_fusion.pt"
)

pd.DataFrame(history_D).to_csv(
    output_dir / "model_D_training_history.csv",
    index=False
)

print("\n=== ALL FOUR VALIDATION RESULTS ===")

results = pd.DataFrame([
    {
        "Model": "A_baseline",
        "Proteomics": False,
        "Sex_aware": False,
        "Best_val_RMSE": best_val_A
    },
    {
        "Model": "B_proteomics",
        "Proteomics": True,
        "Sex_aware": False,
        "Best_val_RMSE": best_val_B
    },
    {
        "Model": "C_sex_aware",
        "Proteomics": False,
        "Sex_aware": True,
        "Best_val_RMSE": best_val_C
    },
    {
        "Model": "D_full",
        "Proteomics": True,
        "Sex_aware": True,
        "Best_val_RMSE": best_val_D
    }
])

results = results.sort_values("Best_val_RMSE")

display(results)

results.to_csv(
    output_dir / "four_model_validation_comparison.csv",
    index=False
)

print("\nModel D checkpoint saved!")
print("Validation comparison saved!")


Training Model D...
Modalities: ['expression', 'mutation', 'cnv', 'proteomics']
Sex-aware input: True
Epoch 01 | Train RMSE: 1.8411 | Val RMSE: 1.4053
Epoch 02 | Train RMSE: 1.3072 | Val RMSE: 1.3466
Epoch 03 | Train RMSE: 1.2589 | Val RMSE: 1.3576
Epoch 04 | Train RMSE: 1.2259 | Val RMSE: 1.3719
Epoch 05 | Train RMSE: 1.2097 | Val RMSE: 1.3499
Epoch 06 | Train RMSE: 1.1929 | Val RMSE: 1.3400
Epoch 07 | Train RMSE: 1.1812 | Val RMSE: 1.3674
Epoch 08 | Train RMSE: 1.1659 | Val RMSE: 1.3437
Epoch 09 | Train RMSE: 1.1576 | Val RMSE: 1.3509
Epoch 10 | Train RMSE: 1.1510 | Val RMSE: 1.3666
Early stopping triggered.

=== ALL FOUR VALIDATION RESULTS ===


,Model,Proteomics,Sex_aware,Best_val_RMSE
0,A_baseline,False,False,1.295261
2,C_sex_aware,False,True,1.318074
1,B_proteomics,True,False,1.327192
3,D_full,True,True,1.340042



Model D checkpoint saved!
Validation comparison saved!


In [20]:

import numpy as np
import pandas as pd
import torch
from pathlib import Path
from scipy.stats import pearsonr

# Evaluate without changing model parameters
def evaluate_model_by_sex(model, loader, model_name):

    model = model.to(device)
    model.eval()

    predictions = []
    targets_all = []
    sex_all = []

    with torch.no_grad():
        for cells, drugs, sexes, targets in loader:
            cells = cells.to(device)
            drugs = drugs.to(device)
            sexes = sexes.to(device)

            batch_omics = {
                name: values[cells]
                for name, values in omics_device.items()
                if name in model.modalities
            }

            pred = model(
                batch_omics,
                drugs,
                sexes if model.use_sex else None
            )

            predictions.extend(pred.cpu().numpy())
            targets_all.extend(targets.numpy())
            sex_all.extend(sexes.cpu().numpy())

    df = pd.DataFrame({
        "actual": targets_all,
        "predicted": predictions,
        "sex_idx": sex_all
    })

    df["sex"] = df["sex_idx"].map({
        0: "Female",
        1: "Male"
    })

    results = []

    for group, subset in [
        ("Overall", df),
        ("Female", df[df["sex"] == "Female"]),
        ("Male", df[df["sex"] == "Male"])
    ]:
        y = subset["actual"].to_numpy()
        p = subset["predicted"].to_numpy()

        rmse = np.sqrt(np.mean((y - p) ** 2))
        mae = np.mean(np.abs(y - p))

        if len(y) > 2 and np.std(y) > 0 and np.std(p) > 0:
            corr = pearsonr(y, p).statistic
        else:
            corr = np.nan

        results.append({
            "Model": model_name,
            "Group": group,
            "N_pairs": len(subset),
            "RMSE": float(rmse),
            "MAE": float(mae),
            "Pearson": float(corr)
        })

    return pd.DataFrame(results), df


# Evaluate each of the four trained configurations
trained_models = {
    "A_baseline": model_A,
    "B_proteomics": model_B,
    "C_sex_aware": model_C,
    "D_full": model_D
}

all_metrics = []
all_predictions = {}

for name, model in trained_models.items():

    metrics_df, predictions_df = evaluate_model_by_sex(
        model,
        test_loader,
        name
    )

    all_metrics.append(metrics_df)
    all_predictions[name] = predictions_df

# Combine results
test_metrics = pd.concat(
    all_metrics,
    ignore_index=True
)

print("=== HELD-OUT TEST PERFORMANCE ===")
display(
    test_metrics.round(4)
)

# Calculate female-minus-male RMSE gap
gap_table = test_metrics.pivot(
    index="Model",
    columns="Group",
    values="RMSE"
)

gap_table["Female_minus_Male_RMSE"] = (
    gap_table["Female"] - gap_table["Male"]
)

print("\n=== SEX-STRATIFIED RMSE GAPS ===")
display(
    gap_table.round(4)
)

# Save results
output_dir = Path("/content/week3_outputs")
output_dir.mkdir(exist_ok=True)

test_metrics.to_csv(
    output_dir / "four_model_test_metrics_by_sex.csv",
    index=False
)

gap_table.to_csv(
    output_dir / "four_model_test_rmse_gaps.csv"
)

print("\nTest results saved successfully!")


=== HELD-OUT TEST PERFORMANCE ===


,Model,Group,N_pairs,RMSE,MAE,Pearson
0,A_baseline,Overall,15965,1.3675,1.0256,0.8609
1,A_baseline,Female,7267,1.3747,1.0294,0.8637
2,A_baseline,Male,8698,1.3614,1.0224,0.8590
3,B_proteomics,Overall,15965,1.3741,1.0200,0.8595
4,B_proteomics,Female,7267,1.3682,1.0068,0.8648
5,B_proteomics,Male,8698,1.3791,1.0310,0.8552
6,C_sex_aware,Overall,15965,1.3757,1.0262,0.8584
7,C_sex_aware,Female,7267,1.3779,1.0210,0.8637
8,C_sex_aware,Male,8698,1.3738,1.0305,0.8561
9,D_full,Overall,15965,1.3495,1.0082,0.8646



=== SEX-STRATIFIED RMSE GAPS ===


Group,Female,Male,Overall,Female_minus_Male_RMSE
Model,,,,
A_baseline,1.3747,1.3614,1.3675,0.0132
B_proteomics,1.3682,1.3791,1.3741,-0.0109
C_sex_aware,1.3779,1.3738,1.3757,0.0041
D_full,1.3398,1.3576,1.3495,-0.0178



Test results saved successfully!


In [21]:

import numpy as np
import pandas as pd
from pathlib import Path

# Recover the test observations in DataLoader order
test_rows = model_pairs[
    model_pairs["split"] == "test"
].reset_index(drop=True)

# Verify prediction alignment
for name, pred_df in all_predictions.items():
    assert len(pred_df) == len(test_rows)
    assert np.allclose(
        pred_df["actual"].to_numpy(),
        test_rows["response"].to_numpy(),
        atol=1e-5
    ), f"Prediction ordering mismatch for {name}"

def bootstrap_sex_gap(df, n_boot=1000, seed=42):
    rng = np.random.default_rng(seed)

    # Keep entire cell-line response groups together
    groups = {
        sidm: group
        for sidm, group in df.groupby("SIDM")
    }

    female_ids = df.loc[
        df["sex"] == "Female", "SIDM"
    ].unique()

    male_ids = df.loc[
        df["sex"] == "Male", "SIDM"
    ].unique()

    gaps = []

    for _ in range(n_boot):
        female_sample = rng.choice(
            female_ids, size=len(female_ids), replace=True
        )
        male_sample = rng.choice(
            male_ids, size=len(male_ids), replace=True
        )

        female_df = pd.concat(
            [groups[s] for s in female_sample],
            ignore_index=True
        )
        male_df = pd.concat(
            [groups[s] for s in male_sample],
            ignore_index=True
        )

        female_rmse = np.sqrt(
            np.mean(
                (female_df["actual"] -
                 female_df["predicted"]) ** 2
            )
        )

        male_rmse = np.sqrt(
            np.mean(
                (male_df["actual"] -
                 male_df["predicted"]) ** 2
            )
        )

        gaps.append(female_rmse - male_rmse)

    return np.percentile(gaps, [2.5, 50, 97.5])


bootstrap_results = []

for name, pred_df in all_predictions.items():

    # Add original cell-line identifiers
    combined = pred_df.copy()
    combined["SIDM"] = test_rows["SIDM"].values

    lower, median, upper = bootstrap_sex_gap(
        combined,
        n_boot=1000,
        seed=42
    )

    bootstrap_results.append({
        "Model": name,
        "Median_gap": median,
        "CI_lower": lower,
        "CI_upper": upper,
        "CI_includes_zero": lower <= 0 <= upper
    })

bootstrap_df = pd.DataFrame(bootstrap_results)

print("=== CELL-LINE BOOTSTRAP RESULTS ===")
display(bootstrap_df.round(4))

output_dir = Path("/content/week3_outputs")

bootstrap_df.to_csv(
    output_dir / "sex_rmse_gap_bootstrap.csv",
    index=False
)

print("\nSaved: sex_rmse_gap_bootstrap.csv")


=== CELL-LINE BOOTSTRAP RESULTS ===


,Model,Median_gap,CI_lower,CI_upper,CI_includes_zero
0,A_baseline,0.0105,-0.1445,0.1704,True
1,B_proteomics,-0.0135,-0.1704,0.1469,True
2,C_sex_aware,0.0018,-0.1343,0.1508,True
3,D_full,-0.0196,-0.1672,0.1390,True



Saved: sex_rmse_gap_bootstrap.csv


In [22]:

import numpy as np
import pandas as pd
from pathlib import Path

# Verify both prediction tables use the same observation order
pred_A = all_predictions["A_baseline"]
pred_D = all_predictions["D_full"]

assert len(pred_A) == len(pred_D) == len(test_rows)

assert np.allclose(
    pred_A["actual"].to_numpy(),
    pred_D["actual"].to_numpy()
)

# Build a paired prediction table
paired = pd.DataFrame({
    "SIDM": test_rows["SIDM"].to_numpy(),
    "sex": test_rows["final_sex"].to_numpy(),
    "actual": pred_A["actual"].to_numpy(),
    "pred_A": pred_A["predicted"].to_numpy(),
    "pred_D": pred_D["predicted"].to_numpy()
})

def paired_bootstrap_difference(
    df, n_boot=1000, seed=42
):
    rng = np.random.default_rng(seed)

    groups = {
        sidm: group
        for sidm, group in df.groupby("SIDM")
    }

    cell_ids = np.array(list(groups.keys()))
    differences = []

    for _ in range(n_boot):
        sampled = rng.choice(
            cell_ids,
            size=len(cell_ids),
            replace=True
        )

        boot = pd.concat(
            [groups[s] for s in sampled],
            ignore_index=True
        )

        rmse_A = np.sqrt(
            np.mean(
                (boot["actual"] - boot["pred_A"]) ** 2
            )
        )

        rmse_D = np.sqrt(
            np.mean(
                (boot["actual"] - boot["pred_D"]) ** 2
            )
        )

        differences.append(rmse_D - rmse_A)

    return {
        "Observed_difference": (
            np.sqrt(np.mean((df["actual"] - df["pred_D"]) ** 2))
            - np.sqrt(np.mean((df["actual"] - df["pred_A"]) ** 2))
        ),
        "CI_lower": np.percentile(differences, 2.5),
        "CI_upper": np.percentile(differences, 97.5)
    }

comparison_rows = []

for group, subset in [
    ("Overall", paired),
    ("Female", paired[paired["sex"] == "Female"]),
    ("Male", paired[paired["sex"] == "Male"])
]:
    result = paired_bootstrap_difference(subset)

    comparison_rows.append({
        "Group": group,
        **result,
        "CI_includes_zero": (
            result["CI_lower"] <= 0 <= result["CI_upper"]
        )
    })

paired_results = pd.DataFrame(comparison_rows)

print("=== MODEL D MINUS MODEL A RMSE ===")
display(paired_results.round(4))

output_dir = Path("/content/week3_outputs")

paired_results.to_csv(
    output_dir / "model_D_vs_A_bootstrap.csv",
    index=False
)

print("\nSaved: model_D_vs_A_bootstrap.csv")


=== MODEL D MINUS MODEL A RMSE ===


,Group,Observed_difference,CI_lower,CI_upper,CI_includes_zero
0,Overall,-0.0179,-0.0434,0.0133,True
1,Female,-0.0349,-0.0667,-0.0061,False
2,Male,-0.0038,-0.0460,0.0394,True



Saved: model_D_vs_A_bootstrap.csv


In [23]:

from pathlib import Path
import zipfile

output_dir = Path("/content/week3_outputs")
zip_path = Path("/content/week3_preliminary_results.zip")

# Collect saved checkpoints and results
files_to_save = sorted(
    p for p in output_dir.rglob("*")
    if p.is_file()
)

with zipfile.ZipFile(
    zip_path, "w", zipfile.ZIP_DEFLATED
) as z:
    for path in files_to_save:
        z.write(
            path,
            arcname=str(path.relative_to(output_dir))
        )

print("=== WEEK 3 PRELIMINARY PACKAGE ===")
print("Files included:", len(files_to_save))

for path in files_to_save:
    print("-", path.relative_to(output_dir))

print(
    "\nZIP size (MB):",
    round(zip_path.stat().st_size / (1024**2), 2)
)

print("Package created:", zip_path)


=== WEEK 3 PRELIMINARY PACKAGE ===
Files included: 16
- eligible_cell_line_mapping.csv
- four_model_test_metrics_by_sex.csv
- four_model_test_rmse_gaps.csv
- four_model_validation_comparison.csv
- matched_drug_response_pairs.csv
- model_A_baseline.pt
- model_A_training_history.csv
- model_B_proteomics.pt
- model_B_training_history.csv
- model_C_sex_aware.pt
- model_C_training_history.csv
- model_D_full_fusion.pt
- model_D_training_history.csv
- model_D_vs_A_bootstrap.csv
- response_pairs_temporary_split.csv
- sex_rmse_gap_bootstrap.csv

ZIP size (MB): 85.5
Package created: /content/week3_preliminary_results.zip


In [24]:

from google.colab import files

files.download(
    "/content/week3_preliminary_results.zip"
)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>